# Ticker Orbit — the market on a globe

Satellite globe with 3D terrain, a marker on every headquarters, a company drawer with the charts a
trader actually reads, and a board for slicing. Scales from a handful of tickers to the whole S&P 500.

**Globe** — Esri satellite / dark / topo basemaps, terrarium elevation for real 3D relief, a galaxy
behind it, and idle rotation. Click a dot to fly to the building; click a cluster to break it apart.
`Full screen` (or `f`) takes the whole thing edge to edge. No API key anywhere in the stack.

**Company drawer** — price with moving averages, Bollinger bands and volume; cumulative return against
the benchmark; MACD and RSI; drawdown and a monthly-return heatmap; market cap, P/E, dividend yield,
beta, headcount and address.

**Board** — sortable leaderboard, risk-versus-return scatter sized by market cap, correlation matrix,
sector weights, and an equal-weight basket of whatever is currently selected.

**Slicers** — search box, sector chips, and 1M / 6M / YTD / 1Y / 3Y / 5Y. Everything recomputes in the
browser from the raw series, so nothing re-fetches when you change the window.


In [ ]:
# Requirements: yfinance, pandas, numpy, lxml (lxml is only for reading the index membership table)
# !pip install -q yfinance pandas numpy lxml

import json, time, math, hashlib, datetime as dt
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option('display.width', 150)
print('pandas', pd.__version__, '| numpy', np.__version__, '| yfinance', yf.__version__)

## 1 · Configuration

`UNIVERSE` is the only dial that matters. `'starter'` is the original nine plus the benchmark,
`'top100'` is the hundred largest US companies, `'sp500'` is the whole index.

Daily bars for every company in the S&P 500 would be a 10 MB payload, so the notebook tiers it:
daily history for the largest `DETAIL_N` companies, weekly for everyone else. Weekly is also computed
for *all* companies, which is what the board uses so every row is comparable. The drawer uses daily
where it has it and says so when it doesn't.

In [ ]:
UNIVERSE   = 'top100'      # 'starter' | 'top100' | 'sp500'
PERIOD     = '5y'
BENCHMARK  = 'SPY'
RISK_FREE  = 0.045
DETAIL_N   = 120           # companies that get daily bars and a full profile
FETCH_CAPS = True          # market cap for everyone (threaded, ~30-60s for 500)
BATCH      = 60            # tickers per yfinance request
OUT_HTML   = 'ticker_orbit.html'
GEO_CACHE  = Path('hq_coords_cache.json')

STARTER = ['BRK-B', 'PGR', 'ALL', 'JPM', 'GS', 'BLK', 'PFE', 'JNJ', 'LLY']

# Ranked roughly by market cap — used to pick the universe for 'top100' and the detail tier for 'sp500'.
MEGACAP = '''NVDA AAPL MSFT GOOGL AMZN META AVGO TSLA BRK-B LLY JPM WMT V UNH XOM MA ORCL COST PG JNJ
HD ABBV NFLX BAC KO CRM CVX AMD PEP TMO MRK LIN ADBE WFC CSCO ACN MCD ABT NOW GE DIS PM IBM TXN QCOM
CAT VZ INTU ISRG GS DHR AXP CMCSA RTX PFE AMGN NEE T SPGI UNP LOW BLK HON ETN COP SYK BKNG TJX PGR
LMT VRTX C BSX MDT ADP UPS MU PANW SCHW ADI FI MMC GILD CB PLD DE SBUX MDLZ LRCX BMY INTC REGN KLAC
SO ELV CI MO ICE DUK EQIX ZTS APH SHW ITW CME WM MCK NOC TGT CL GD EOG MSI PYPL SNPS CDNS AON FDX
ORLY APD HCA EMR NSC PSA ROP MAR SLB'''.split()

print(f'universe: {UNIVERSE} · benchmark: {BENCHMARK} · {PERIOD} of history')

## 2 · Who's in the index

Membership, sector and headquarters city come from the S&P 500 constituents table on Wikipedia, with
a mirrored CSV as a fallback if Wikipedia is unreachable.

In [ ]:
SP500_WIKI = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
SP500_CSV  = 'https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv'

def sp500_table():
    try:
        tables = pd.read_html(SP500_WIKI, storage_options={'User-Agent': 'Mozilla/5.0'})
        df = next(t for t in tables if 'Symbol' in t.columns and 'GICS Sector' in t.columns)
        print(f'index membership from Wikipedia: {len(df)} rows')
    except Exception as e:
        print(f'Wikipedia unavailable ({type(e).__name__}) — falling back to the mirrored CSV')
        df = pd.read_csv(SP500_CSV)
        print(f'index membership from CSV mirror: {len(df)} rows')
    df = df.rename(columns={'Symbol': 'ticker', 'Security': 'name', 'GICS Sector': 'sector',
                            'GICS Sub-Industry': 'industry', 'Headquarters Location': 'hq'})
    df['ticker'] = df['ticker'].str.replace('.', '-', regex=False)   # BRK.B -> BRK-B for Yahoo
    return df[['ticker', 'name', 'sector', 'industry', 'hq']].drop_duplicates('ticker')

index_df = sp500_table()

if UNIVERSE == 'starter':
    keep = STARTER
elif UNIVERSE == 'top100':
    keep = [t for t in MEGACAP if t in set(index_df.ticker)][:100]
else:
    keep = list(index_df.ticker)

universe = index_df[index_df.ticker.isin(keep)].copy()
universe['rank'] = universe.ticker.map({t: i for i, t in enumerate(MEGACAP)}).fillna(9999)
universe = universe.sort_values('rank').reset_index(drop=True)

# the benchmark rides along as a row of its own
universe = pd.concat([universe, pd.DataFrame([{
    'ticker': BENCHMARK, 'name': 'S&P 500 ETF', 'sector': 'Benchmark',
    'industry': 'Index fund', 'hq': 'New York, New York', 'rank': -1}])], ignore_index=True)

TICKERS = list(universe.ticker)
DETAIL  = [t for t in TICKERS if t != BENCHMARK][:DETAIL_N] + [BENCHMARK]
print(f'{len(TICKERS)-1} companies + benchmark · daily history for {len(DETAIL)-1} of them')
universe.head(8)

## 3 · Putting them on the globe

The membership table gives a city, not coordinates. A built-in gazetteer covers the cities where
American corporate headquarters actually cluster; anything it misses falls back to the state centroid,
and then to a one-per-second OpenStreetMap lookup that gets cached to disk so it only ever happens once.
Companies sharing a city get a few hundred metres of deterministic jitter so they don't stack into one dot.

In [ ]:
CITIES = {
 'new york, new york':(40.7128,-74.0060),'brooklyn, new york':(40.6782,-73.9442),
 'purchase, new york':(41.0400,-73.7154),'armonk, new york':(41.1265,-73.7140),
 'rye, new york':(40.9807,-73.6837),'harrison, new york':(40.9690,-73.7126),
 'white plains, new york':(41.0340,-73.7629),'tarrytown, new york':(41.0762,-73.8587),
 'melville, new york':(40.7934,-73.4151),'jericho, new york':(40.7923,-73.5390),
 'buffalo, new york':(42.8864,-78.8784),'rochester, new york':(43.1566,-77.6088),
 'boston, massachusetts':(42.3601,-71.0589),'cambridge, massachusetts':(42.3736,-71.1097),
 'waltham, massachusetts':(42.3765,-71.2356),'marlborough, massachusetts':(42.3459,-71.5523),
 'natick, massachusetts':(42.2835,-71.3495),'needham, massachusetts':(42.2809,-71.2378),
 'framingham, massachusetts':(42.2793,-71.4162),'westborough, massachusetts':(42.2695,-71.6162),
 'andover, massachusetts':(42.6583,-71.1368),'springfield, massachusetts':(42.1015,-72.5898),
 'hartford, connecticut':(41.7658,-72.6734),'stamford, connecticut':(41.0534,-73.5387),
 'norwalk, connecticut':(41.1176,-73.4079),'greenwich, connecticut':(41.0262,-73.6282),
 'danbury, connecticut':(41.3948,-73.4540),'shelton, connecticut':(41.3165,-73.0932),
 'new haven, connecticut':(41.3083,-72.9279),'providence, rhode island':(41.8240,-71.4128),
 'smithfield, rhode island':(41.9223,-71.5495),'woonsocket, rhode island':(42.0029,-71.5147),
 'philadelphia, pennsylvania':(39.9526,-75.1652),'pittsburgh, pennsylvania':(40.4406,-79.9959),
 'radnor, pennsylvania':(40.0459,-75.3596),'wayne, pennsylvania':(40.0443,-75.3874),
 'malvern, pennsylvania':(40.0362,-75.5138),'conshohocken, pennsylvania':(40.0793,-75.3016),
 'allentown, pennsylvania':(40.6084,-75.4902),'hershey, pennsylvania':(40.2859,-76.6502),
 'camp hill, pennsylvania':(40.2398,-76.9233),'newark, new jersey':(40.7357,-74.1724),
 'jersey city, new jersey':(40.7282,-74.0776),'new brunswick, new jersey':(40.4862,-74.4518),
 'princeton, new jersey':(40.3573,-74.6672),'basking ridge, new jersey':(40.7062,-74.5490),
 'bedminster, new jersey':(40.6690,-74.6427),'madison, new jersey':(40.7598,-74.4171),
 'florham park, new jersey':(40.7876,-74.3882),'morris plains, new jersey':(40.8215,-74.4796),
 'roseland, new jersey':(40.8201,-74.2937),'parsippany, new jersey':(40.8579,-74.4260),
 'warren, new jersey':(40.6285,-74.5165),'summit, new jersey':(40.7156,-74.3646),
 'camden, new jersey':(39.9259,-75.1196),'wilmington, delaware':(39.7391,-75.5398),
 'baltimore, maryland':(39.2904,-76.6122),'bethesda, maryland':(38.9847,-77.0947),
 'columbia, maryland':(39.2037,-76.8610),'rockville, maryland':(39.0840,-77.1528),
 'gaithersburg, maryland':(39.1434,-77.2014),'north bethesda, maryland':(39.0440,-77.1200),
 'washington, d.c.':(38.9072,-77.0369),'washington, district of columbia':(38.9072,-77.0369),
 'mclean, virginia':(38.9339,-77.1773),'arlington, virginia':(38.8816,-77.0910),
 'richmond, virginia':(37.5407,-77.4360),'falls church, virginia':(38.8823,-77.1711),
 'reston, virginia':(38.9586,-77.3570),'herndon, virginia':(38.9696,-77.3861),
 'norfolk, virginia':(36.8508,-76.2859),'glen allen, virginia':(37.6659,-77.5083),
 'charlotte, north carolina':(35.2271,-80.8431),'raleigh, north carolina':(35.7796,-78.6382),
 'durham, north carolina':(35.9940,-78.8986),'winston-salem, north carolina':(36.0999,-80.2442),
 'atlanta, georgia':(33.7490,-84.3880),'alpharetta, georgia':(34.0754,-84.2941),
 'duluth, georgia':(34.0029,-84.1446),'sandy springs, georgia':(33.9304,-84.3733),
 'augusta, georgia':(33.4735,-81.9748),'miami, florida':(25.7617,-80.1918),
 'miami beach, florida':(25.7907,-80.1300),'tampa, florida':(27.9506,-82.4572),
 'orlando, florida':(28.5383,-81.3792),'jacksonville, florida':(30.3322,-81.6557),
 'palm beach gardens, florida':(26.8234,-80.1387),'boca raton, florida':(26.3683,-80.1289),
 'naples, florida':(26.1420,-81.7948),'fort lauderdale, florida':(26.1224,-80.1373),
 'st. petersburg, florida':(27.7676,-82.6403),'deerfield beach, florida':(26.3184,-80.0998),
 'nashville, tennessee':(36.1627,-86.7816),'memphis, tennessee':(35.1495,-90.0490),
 'knoxville, tennessee':(35.9606,-83.9207),'chattanooga, tennessee':(35.0456,-85.3097),
 'brentwood, tennessee':(36.0331,-86.7828),'franklin, tennessee':(35.9251,-86.8689),
 'birmingham, alabama':(33.5186,-86.8104),'louisville, kentucky':(38.2527,-85.7585),
 'cincinnati, ohio':(39.1031,-84.5120),'cleveland, ohio':(41.4993,-81.6944),
 'columbus, ohio':(39.9612,-82.9988),'dublin, ohio':(40.0992,-83.1141),
 'akron, ohio':(41.0814,-81.5190),'mayfield village, ohio':(41.5461,-81.4257),
 'westlake, ohio':(41.4553,-81.9179),'findlay, ohio':(41.0442,-83.6499),
 'toledo, ohio':(41.6528,-83.5379),'detroit, michigan':(42.3314,-83.0458),
 'dearborn, michigan':(42.3223,-83.1763),'midland, michigan':(43.6156,-84.2472),
 'grand rapids, michigan':(42.9634,-85.6681),'kalamazoo, michigan':(42.2917,-85.5872),
 'ann arbor, michigan':(42.2808,-83.7430),'chicago, illinois':(41.8781,-87.6298),
 'north chicago, illinois':(42.3256,-87.8612),'deerfield, illinois':(42.1711,-87.8445),
 'northbrook, illinois':(42.1275,-87.8290),'glenview, illinois':(42.0800,-87.8120),
 'lake forest, illinois':(42.2586,-87.8406),'abbott park, illinois':(42.3186,-87.8898),
 'oak brook, illinois':(41.8369,-87.9290),'downers grove, illinois':(41.8089,-88.0112),
 'naperville, illinois':(41.7508,-88.1535),'rosemont, illinois':(41.9953,-87.8845),
 'schaumburg, illinois':(42.0334,-88.0834),'peoria, illinois':(40.6936,-89.5890),
 'moline, illinois':(41.5067,-90.5151),'milwaukee, wisconsin':(43.0389,-87.9065),
 'madison, wisconsin':(43.0731,-89.4012),'minneapolis, minnesota':(44.9778,-93.2650),
 'saint paul, minnesota':(44.9537,-93.0900),'st. paul, minnesota':(44.9537,-93.0900),
 'minnetonka, minnesota':(44.9211,-93.4687),'eden prairie, minnesota':(44.8547,-93.4708),
 'golden valley, minnesota':(44.9861,-93.3663),'richfield, minnesota':(44.8766,-93.2830),
 'st. louis, missouri':(38.6270,-90.1994),'saint louis, missouri':(38.6270,-90.1994),
 'kansas city, missouri':(39.0997,-94.5786),'springfield, missouri':(37.2090,-93.2923),
 'des moines, iowa':(41.5868,-93.6250),'omaha, nebraska':(41.2565,-95.9345),
 'lincoln, nebraska':(40.8136,-96.7026),'little rock, arkansas':(34.7465,-92.2896),
 'bentonville, arkansas':(36.3729,-94.2088),'oklahoma city, oklahoma':(35.4676,-97.5164),
 'tulsa, oklahoma':(36.1540,-95.9928),'dallas, texas':(32.7767,-96.7970),
 'irving, texas':(32.8140,-96.9489),'plano, texas':(33.0198,-96.6989),
 'fort worth, texas':(32.7555,-97.3308),'houston, texas':(29.7604,-95.3698),
 'spring, texas':(30.0799,-95.4172),'the woodlands, texas':(30.1658,-95.4613),
 'san antonio, texas':(29.4241,-98.4936),'austin, texas':(30.2672,-97.7431),
 'round rock, texas':(30.5083,-97.6789),'el paso, texas':(31.7619,-106.4850),
 'richardson, texas':(32.9483,-96.7299),'denver, colorado':(39.7392,-104.9903),
 'englewood, colorado':(39.6478,-104.9878),'greenwood village, colorado':(39.6172,-104.9508),
 'broomfield, colorado':(39.9205,-105.0867),'boulder, colorado':(40.0150,-105.2705),
 'colorado springs, colorado':(38.8339,-104.8214),'salt lake city, utah':(40.7608,-111.8910),
 'phoenix, arizona':(33.4484,-112.0740),'scottsdale, arizona':(33.4942,-111.9261),
 'chandler, arizona':(33.3062,-111.8413),'tempe, arizona':(33.4255,-111.9400),
 'albuquerque, new mexico':(35.0844,-106.6504),'las vegas, nevada':(36.1699,-115.1398),
 'reno, nevada':(39.5296,-119.8138),'seattle, washington':(47.6062,-122.3321),
 'redmond, washington':(47.6740,-122.1215),'bellevue, washington':(47.6101,-122.2015),
 'issaquah, washington':(47.5301,-122.0326),'renton, washington':(47.4829,-122.2171),
 'portland, oregon':(45.5152,-122.6784),'beaverton, oregon':(45.4871,-122.8037),
 'hillsboro, oregon':(45.5229,-122.9898),'san francisco, california':(37.7749,-122.4194),
 'san jose, california':(37.3382,-121.8863),'santa clara, california':(37.3541,-121.9552),
 'cupertino, california':(37.3229,-122.0322),'mountain view, california':(37.3861,-122.0839),
 'palo alto, california':(37.4419,-122.1430),'menlo park, california':(37.4530,-122.1817),
 'sunnyvale, california':(37.3688,-122.0363),'redwood city, california':(37.4852,-122.2364),
 'foster city, california':(37.5585,-122.2711),'san mateo, california':(37.5630,-122.3255),
 'milpitas, california':(37.4323,-121.8996),'fremont, california':(37.5485,-121.9886),
 'pleasanton, california':(37.6624,-121.8747),'san ramon, california':(37.7799,-121.9780),
 'dublin, california':(37.7022,-121.9358),'emeryville, california':(37.8313,-122.2852),
 'oakland, california':(37.8044,-122.2712),'berkeley, california':(37.8715,-122.2730),
 'los angeles, california':(34.0522,-118.2437),'santa monica, california':(34.0195,-118.4912),
 'culver city, california':(34.0211,-118.3965),'el segundo, california':(33.9192,-118.4165),
 'burbank, california':(34.1808,-118.3090),'glendale, california':(34.1425,-118.2551),
 'pasadena, california':(34.1478,-118.1445),'thousand oaks, california':(34.1706,-118.8376),
 'irvine, california':(33.6846,-117.8265),'costa mesa, california':(33.6411,-117.9187),
 'newport beach, california':(33.6189,-117.9298),'san diego, california':(32.7157,-117.1611),
 'carlsbad, california':(33.1581,-117.3506),'foothill ranch, california':(33.6853,-117.6656),
 'santa barbara, california':(34.4208,-119.6982),'sacramento, california':(38.5816,-121.4944),
 'honolulu, hawaii':(21.3069,-157.8583),'anchorage, alaska':(61.2181,-149.9003),
 'boise, idaho':(43.6150,-116.2023),'billings, montana':(45.7833,-108.5007),
 'sioux falls, south dakota':(43.5460,-96.7313),'fargo, north dakota':(46.8772,-96.7898),
 'wichita, kansas':(37.6872,-97.3301),'overland park, kansas':(38.9822,-94.6708),
 'new orleans, louisiana':(29.9511,-90.0715),'baton rouge, louisiana':(30.4515,-91.1871),
 'jackson, mississippi':(32.2988,-90.1848),'columbia, south carolina':(34.0007,-81.0348),
 'greenville, south carolina':(34.8526,-82.3940),'charleston, south carolina':(32.7765,-79.9311),
 'portland, maine':(43.6591,-70.2568),'manchester, new hampshire':(42.9956,-71.4548),
 'burlington, vermont':(44.4759,-73.2121),'charleston, west virginia':(38.3498,-81.6326),
 'indianapolis, indiana':(39.7684,-86.1581),'carmel, indiana':(39.9784,-86.1180),
 'fort wayne, indiana':(41.0793,-85.1394),'evansville, indiana':(37.9716,-87.5711),
 'warsaw, indiana':(41.2381,-85.8530),
 'dublin, ireland':(53.3498,-6.2603),'cork, ireland':(51.8985,-8.4756),
 'hamilton, bermuda':(32.2949,-64.7814),'pembroke, bermuda':(32.3000,-64.7900),
 'london, united kingdom':(51.5074,-0.1278),'zurich, switzerland':(47.3769,8.5417),
 'schaffhausen, switzerland':(47.6960,8.6340),'amsterdam, netherlands':(52.3676,4.9041),
 'singapore':(1.3521,103.8198),'toronto, canada':(43.6532,-79.3832),
}

STATES = {
 'alabama':(32.8,-86.8),'alaska':(64.0,-152.0),'arizona':(34.3,-111.7),'arkansas':(34.9,-92.4),
 'california':(37.2,-119.5),'colorado':(39.0,-105.5),'connecticut':(41.6,-72.7),'delaware':(39.0,-75.5),
 'florida':(28.6,-82.4),'georgia':(32.6,-83.4),'hawaii':(20.3,-156.4),'idaho':(44.4,-114.6),
 'illinois':(40.0,-89.2),'indiana':(39.9,-86.3),'iowa':(42.0,-93.5),'kansas':(38.5,-98.4),
 'kentucky':(37.5,-85.3),'louisiana':(31.1,-92.0),'maine':(45.4,-69.2),'maryland':(39.0,-76.8),
 'massachusetts':(42.3,-71.8),'michigan':(44.3,-85.4),'minnesota':(46.3,-94.3),'mississippi':(32.7,-89.7),
 'missouri':(38.4,-92.5),'montana':(47.0,-109.6),'nebraska':(41.5,-99.8),'nevada':(39.3,-116.6),
 'new hampshire':(43.7,-71.6),'new jersey':(40.2,-74.7),'new mexico':(34.4,-106.1),'new york':(42.9,-75.5),
 'north carolina':(35.5,-79.4),'north dakota':(47.4,-100.5),'ohio':(40.3,-82.8),'oklahoma':(35.6,-97.5),
 'oregon':(43.9,-120.6),'pennsylvania':(40.9,-77.8),'rhode island':(41.7,-71.6),'south carolina':(33.9,-80.9),
 'south dakota':(44.4,-100.2),'tennessee':(35.8,-86.4),'texas':(31.5,-99.3),'utah':(39.3,-111.7),
 'vermont':(44.1,-72.7),'virginia':(37.5,-78.9),'washington':(47.4,-120.4),'west virginia':(38.6,-80.6),
 'wisconsin':(44.6,-89.7),'wyoming':(43.0,-107.5),'district of columbia':(38.9,-77.0),
}

geo_cache = json.loads(GEO_CACHE.read_text()) if GEO_CACHE.exists() else {}

def nominatim(place):
    """One-per-second OpenStreetMap lookup, cached to disk. Returns None if it can't resolve."""
    import urllib.request, urllib.parse
    url = 'https://nominatim.openstreetmap.org/search?' + urllib.parse.urlencode(
        {'q': place, 'format': 'json', 'limit': 1})
    req = urllib.request.Request(url, headers={'User-Agent': 'ticker-orbit-notebook/1.0'})
    try:
        time.sleep(1.1)
        with urllib.request.urlopen(req, timeout=12) as r:
            hits = json.load(r)
        return (float(hits[0]['lat']), float(hits[0]['lon'])) if hits else None
    except Exception:
        return None

def jitter(ticker, scale=0.022):
    h = int(hashlib.md5(ticker.encode()).hexdigest()[:8], 16)
    return ((h % 1000) / 1000 - 0.5) * scale, (((h >> 10) % 1000) / 1000 - 0.5) * scale

def locate(place, ticker):
    key = str(place).strip().lower()
    for lookup in (CITIES.get(key), geo_cache.get(key)):
        if lookup:
            dy, dx = jitter(ticker)
            return round(lookup[0] + dy, 5), round(lookup[1] + dx, 5), 'known'
    state = key.split(',')[-1].strip()
    hit = nominatim(place)
    if hit:
        geo_cache[key] = list(hit)
        dy, dx = jitter(ticker)
        return round(hit[0] + dy, 5), round(hit[1] + dx, 5), 'geocoded'
    if state in STATES:
        dy, dx = jitter(ticker, 0.6)
        return round(STATES[state][0] + dy, 5), round(STATES[state][1] + dx, 5), 'state'
    return None, None, 'missing'

located = [locate(r['hq'], r['ticker']) for r in universe.to_dict('records')]
universe['lat']    = [x[0] for x in located]
universe['lon']    = [x[1] for x in located]
universe['geo']    = [x[2] for x in located]
GEO_CACHE.write_text(json.dumps(geo_cache))

print(universe.geo.value_counts().to_string())
missing = universe[universe.lat.isna()]
if len(missing):
    print('\nno coordinates (they stay off the globe but keep all their charts):')
    print(missing[['ticker', 'hq']].to_string(index=False))

## 4 · Prices

Two passes. Weekly bars for everyone, which is what makes 500 companies comparable without a
50 MB payload; daily bars for the detail tier. yfinance's column layout has flipped between
`(field, ticker)` and `(ticker, field)` across versions, so `field()` reads whichever level holds it.

In [ ]:
def field(df, name, want):
    if isinstance(df.columns, pd.MultiIndex):
        for lvl in (0, 1):
            if name in set(df.columns.get_level_values(lvl)):
                out = df.xs(name, axis=1, level=lvl)
                break
        else:
            return pd.DataFrame(index=df.index, columns=want, dtype=float)
    else:
        out = df[[name]].rename(columns={name: want[0]})
    return out.reindex(columns=want)

def fetch(tickers, interval):
    closes, volumes = [], []
    for i in range(0, len(tickers), BATCH):
        chunk = tickers[i:i+BATCH]
        raw = yf.download(chunk, period=PERIOD, interval=interval, auto_adjust=True,
                          progress=False, group_by='column', threads=True)
        if raw.empty:
            print(f'  batch {i//BATCH+1}: empty response'); continue
        closes.append(field(raw, 'Close', chunk))
        volumes.append(field(raw, 'Volume', chunk))
        print(f'  {interval}: {min(i+BATCH, len(tickers))}/{len(tickers)}', end='\r')
    c = pd.concat(closes, axis=1).sort_index()
    v = pd.concat(volumes, axis=1).sort_index().reindex(c.index)
    return c.ffill(), v.fillna(0)

print('weekly bars for the full universe')
close_w, vol_w = fetch(TICKERS, '1wk')
print(f'\n  {close_w.shape[0]} weekly bars, {close_w.shape[1]} tickers')

print('daily bars for the detail tier')
close_d, vol_d = fetch(DETAIL, '1d')
print(f'\n  {close_d.shape[0]} daily bars, {close_d.shape[1]} tickers')

# drop anything with no usable history at all
dead = [t for t in TICKERS if close_w[t].notna().sum() < 12]
if dead:
    print('no history, dropped:', ', '.join(dead))
    TICKERS = [t for t in TICKERS if t not in dead]
    universe = universe[universe.ticker.isin(TICKERS)].reset_index(drop=True)

## 5 · Fundamentals

Market cap for everyone (it drives bubble size, sector weights and which labels win space on the map),
full profiles for the detail tier. Both threaded, both tolerant of Yahoo saying no.

In [ ]:
def fast_cap(ticker):
    try:
        fi = yf.Ticker(ticker).fast_info
        return ticker, (fi.get('market_cap') if hasattr(fi, 'get') else fi['market_cap'])
    except Exception:
        return ticker, None

def full_profile(ticker):
    try:
        info = yf.Ticker(ticker).get_info() or {}
    except Exception:
        return ticker, {}
    dy = info.get('dividendYield')
    return ticker, {
        'marketCap': info.get('marketCap'), 'pe': info.get('trailingPE'),
        'forwardPe': info.get('forwardPE'), 'divYield': (dy/100 if (dy or 0) > 1 else dy),
        'beta': info.get('beta'), 'employees': info.get('fullTimeEmployees'),
        'website': info.get('website'), 'summary': (info.get('longBusinessSummary') or '')[:900] or None,
    }

PROFILES = {t: {'marketCap': None, 'pe': None, 'forwardPe': None, 'divYield': None, 'beta': None,
                'employees': None, 'website': None, 'summary': None,
                'avgVolume': float(vol_w[t].tail(13).mean() / 5) if t in vol_w else None}
            for t in TICKERS}

with ThreadPoolExecutor(max_workers=8) as pool:
    print(f'profiles for {len(DETAIL)} companies...')
    for t, p in pool.map(full_profile, [t for t in DETAIL if t in TICKERS]):
        PROFILES[t].update({k: v for k, v in p.items() if v is not None})
    if FETCH_CAPS:
        rest = [t for t in TICKERS if PROFILES[t]['marketCap'] is None]
        print(f'market caps for the remaining {len(rest)}...')
        for t, c in pool.map(fast_cap, rest):
            PROFILES[t]['marketCap'] = c

have = sum(1 for t in TICKERS if PROFILES[t]['marketCap'])
print(f'market cap resolved for {have}/{len(TICKERS)}')

## 6 · The read, in text

The notebook still answers the question without opening the dashboard. Weekly bars, so every company
in the universe is on the same footing.

In [ ]:
PPY = 52
rets_w = close_w.pct_change()

ann_ret = rets_w.mean() * PPY
ann_vol = rets_w.std() * np.sqrt(PPY)
sharpe  = (ann_ret - RISK_FREE) / ann_vol
bench_r = rets_w[BENCHMARK]
beta    = rets_w.apply(lambda c: c.cov(bench_r) / bench_r.var())

summary = pd.DataFrame({
    'company':    universe.set_index('ticker')['name'],
    'sector':     universe.set_index('ticker')['sector'],
    'last':       close_w.iloc[-1],
    'return_5y':  close_w.ffill().iloc[-1] / close_w.bfill().iloc[0] - 1,
    'ann_return': ann_ret, 'volatility': ann_vol, 'sharpe': sharpe, 'beta': beta,
    'market_cap': pd.Series({t: PROFILES[t]['marketCap'] for t in TICKERS}),
}).dropna(subset=['last']).sort_values('sharpe', ascending=False)

def money(v): return '—' if pd.isna(v) else f'${v/1e9:,.0f}B'
fmt = {'last':'${:,.2f}','return_5y':'{:+.1%}','ann_return':'{:+.1%}','volatility':'{:.1%}',
       'sharpe':'{:.2f}','beta':'{:.2f}'}
print(f'Top 15 by Sharpe, of {len(summary)} companies\n')
print(summary.head(15).to_string(
    formatters={**{k: (lambda v, f=f: f.format(v)) for k, f in fmt.items()}, 'market_cap': money}))

## 7 · Payload

Weekly closes and volumes for every company, daily for the detail tier, plus profile and coordinates.
Volumes ship in millions and prices round to cents — at 500 companies that trimming is worth about a
megabyte. Indicators and window statistics are deliberately left to the browser.

In [ ]:
def arr(series, price=True):
    out = []
    for v in series:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            out.append(None)
        else:
            out.append(round(float(v), 2) if price else round(float(v)/1e6, 2))
    return out

daily_ok = set(close_d.columns)
companies = []
for row in universe.to_dict('records'):
    t = row['ticker']
    if t not in close_w.columns:
        continue
    c = {'ticker': t, 'name': row['name'], 'sector': row['sector'], 'industry': row['industry'],
         'profile': PROFILES[t],
         'w': {'close': arr(close_w[t]), 'volume': arr(vol_w[t], price=False)}}
    if not (pd.isna(row['lat']) or pd.isna(row['lon'])):
        c['hq'] = {'city': row['hq'], 'address': None,
                   'lat': float(row['lat']), 'lon': float(row['lon'])}
    if t in daily_ok and close_d[t].notna().any():
        c['d'] = {'close': arr(close_d[t]), 'volume': arr(vol_d[t], price=False)}
    companies.append(c)

PAYLOAD = {
    'generated': dt.date.today().isoformat(),
    'benchmark': BENCHMARK, 'riskFree': RISK_FREE,
    'datesW': [d.strftime('%Y-%m-%d') for d in close_w.index],
    'datesD': [d.strftime('%Y-%m-%d') for d in close_d.index],
    'companies': companies,
}
payload_json = json.dumps(PAYLOAD, separators=(',', ':'), allow_nan=False)
print(f'{len(companies)} companies · {sum(1 for c in companies if "d" in c)} with daily bars · '
      f'payload {len(payload_json)/1024/1024:.2f} MB')

## 8 · The dashboard

One self-contained HTML document. MapLibre GL draws the globe; basemaps are Esri's World Imagery,
Dark Gray Canvas and Topo; elevation is the AWS terrarium tile set; ECharts draws the panels; the
galaxy is a canvas behind the globe. Every tile source here is key-free — nothing in this file will
ever ask for a token or stamp a watermark on the map.

Restyling starts with the CSS variables at the top of the template.

In [ ]:
TEMPLATE = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1,viewport-fit=cover">
<title>Ticker Orbit</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Archivo:wght@400;500;600;800&family=IBM+Plex+Mono:wght@400;500;600&display=swap" rel="stylesheet">
<link href="https://cdn.jsdelivr.net/npm/maplibre-gl@5.24.0/dist/maplibre-gl.css" rel="stylesheet">
<style>
:root{
  --void:#03050B; --panel:rgba(8,14,28,.86); --panel2:rgba(11,19,37,.95);
  --line:rgba(122,152,204,.20); --line2:rgba(122,152,204,.38);
  --ink:#E7EEFB; --muted:#8CA1C4; --dim:#5C6E90;
  --up:#3DDC97; --down:#FF5F6D; --focus:#F2C14E; --beam:#4FD3E8;
  --rail:250px; --drawer:440px;
  --mono:'IBM Plex Mono',ui-monospace,monospace;
  --sans:'Archivo',system-ui,-apple-system,'Segoe UI',sans-serif;
}
*{box-sizing:border-box}
html,body{height:100%;margin:0;background:var(--void);color:var(--ink);font-family:var(--sans);overflow:hidden}
body{font-size:14px;line-height:1.45}
button,input{font:inherit;color:inherit;background:none;border:none}
button{cursor:pointer}
:focus-visible{outline:2px solid var(--focus);outline-offset:2px}
.num{font-family:var(--mono);font-variant-numeric:tabular-nums}
.up{color:var(--up)} .down{color:var(--down)}

#galaxy{position:fixed;inset:0;z-index:0;display:block}
#globe{position:fixed;inset:0;z-index:1;background:transparent}
#globe canvas{outline:none}
.maplibregl-ctrl-attrib{background:rgba(3,5,11,.65)!important;color:var(--dim)!important;font-size:10px}
.maplibregl-ctrl-attrib a{color:var(--muted)!important}
.maplibregl-ctrl-group{background:var(--panel)!important;border:1px solid var(--line)!important}
.maplibregl-ctrl-group button+button{border-top:1px solid var(--line)!important}
.maplibregl-ctrl-group button span{filter:invert(1) hue-rotate(180deg) brightness(1.4)}
.vignette{position:fixed;inset:0;z-index:2;pointer-events:none;
  background:radial-gradient(125% 95% at 50% 46%,transparent 42%,rgba(1,2,6,.78) 100%)}
.popup .maplibregl-popup-content{background:var(--panel2);border:1px solid var(--line2);border-radius:8px;
  color:var(--ink);font-size:12px;padding:8px 11px;box-shadow:0 10px 30px rgba(0,0,0,.6)}
.popup .maplibregl-popup-tip{border-top-color:var(--panel2)!important;border-bottom-color:var(--panel2)!important}
.popup b{font-family:var(--mono)}

/* ---------------- HUD ---------------- */
.hud{position:fixed;z-index:6;top:0;left:0;right:0;display:flex;gap:14px;align-items:center;flex-wrap:wrap;
  padding:calc(10px + env(safe-area-inset-top,0px)) 16px 10px;
  background:linear-gradient(180deg,rgba(3,5,11,.94),rgba(3,5,11,0))}
.brand{display:flex;flex-direction:column;line-height:1.05}
.brand b{font-weight:800;font-size:19px;letter-spacing:-.02em}
.brand span{font-size:10.5px;color:var(--dim);font-family:var(--mono)}
.tabs{display:flex;border:1px solid var(--line);border-radius:999px;overflow:hidden;background:var(--panel)}
.tabs button{padding:7px 16px;color:var(--muted);font-size:13px}
.tabs button[aria-selected=true]{background:rgba(79,211,232,.15);color:var(--ink)}
.search{display:flex;align-items:center;gap:7px;border:1px solid var(--line);border-radius:8px;
  background:var(--panel);padding:5px 10px;min-width:216px}
.search input{width:100%;font-size:12.5px;font-family:var(--mono);outline:none}
.search input::placeholder{color:var(--dim)}
.slicers{display:flex;gap:10px;align-items:center;margin-left:auto;flex-wrap:wrap;justify-content:flex-end}
.seg{display:flex;gap:2px;background:var(--panel);border:1px solid var(--line);border-radius:8px;padding:3px}
.seg button{padding:5px 10px;border-radius:6px;font-family:var(--mono);font-size:12px;color:var(--muted)}
.seg button[aria-pressed=true]{background:rgba(232,177,76,.16);color:var(--focus)}
.ghost{padding:6px 12px;border:1px solid var(--line);border-radius:8px;background:var(--panel);
  font-size:12px;font-family:var(--mono);color:var(--muted)}
.ghost:hover{color:var(--ink);border-color:var(--line2)}
.ghost[aria-pressed=true]{color:var(--focus);border-color:rgba(242,193,78,.45)}
.sectors{position:fixed;z-index:6;left:16px;top:58px;display:flex;gap:5px;flex-wrap:wrap;max-width:min(62vw,720px)}
.chip{display:flex;align-items:center;gap:6px;padding:4px 10px 4px 8px;border-radius:999px;
  border:1px solid var(--line);background:var(--panel);font-size:11.5px;color:var(--dim)}
.chip[aria-pressed=true]{color:var(--ink)}
.chip .dot{width:7px;height:7px;border-radius:50%;background:var(--c);opacity:.3}
.chip[aria-pressed=true] .dot{opacity:1;box-shadow:0 0 9px var(--c)}

/* ---------------- map controls ---------------- */
.mapctl{position:fixed;z-index:6;left:16px;bottom:34px;display:flex;flex-direction:column;gap:6px;align-items:flex-start}
.mapctl .row{display:flex;gap:4px;background:var(--panel);border:1px solid var(--line);border-radius:8px;padding:3px}
.mapctl button{padding:5px 10px;border-radius:6px;font-size:11.5px;font-family:var(--mono);color:var(--muted)}
.mapctl button[aria-pressed=true]{background:rgba(79,211,232,.18);color:var(--ink)}

/* ---------------- roster ---------------- */
.rail{position:fixed;z-index:5;top:104px;bottom:96px;left:16px;width:var(--rail);
  display:flex;flex-direction:column;gap:7px;overflow-y:auto;padding-right:4px;scrollbar-width:thin}
.rail::-webkit-scrollbar{width:6px} .rail::-webkit-scrollbar-thumb{background:var(--line);border-radius:3px}
.railhead{font-size:10.5px;color:var(--dim);font-family:var(--mono);padding:0 2px}
.card{display:grid;grid-template-columns:1fr auto;gap:1px 8px;align-items:center;width:100%;text-align:left;
  padding:8px 11px;border:1px solid var(--line);border-left:3px solid var(--c,var(--muted));
  border-radius:9px;background:var(--panel);backdrop-filter:blur(7px);transition:transform .14s,border-color .14s}
.card:hover{transform:translateX(3px);border-color:var(--line2)}
.card[aria-current=true]{background:var(--panel2);box-shadow:0 0 0 1px var(--c) inset}
.card .tkr{font-family:var(--mono);font-size:12.5px;font-weight:600}
.card .chg{font-family:var(--mono);font-size:12.5px}
.card .nm{grid-column:1/2;font-size:10.5px;color:var(--dim);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}

/* ---------------- drawer ---------------- */
.drawer{position:fixed;z-index:8;top:0;right:0;bottom:0;width:var(--drawer);max-width:100vw;
  background:var(--panel2);backdrop-filter:blur(14px);border-left:1px solid var(--line);
  transform:translateX(102%);transition:transform .34s cubic-bezier(.22,.8,.3,1);
  display:flex;flex-direction:column;padding-top:env(safe-area-inset-top,0px)}
.drawer.open{transform:none}
.dhead{padding:16px 20px 12px;border-bottom:1px solid var(--line);position:relative}
.dhead .x{position:absolute;top:12px;right:12px;color:var(--dim);font-size:20px;line-height:1;padding:4px 8px}
.dhead .x:hover{color:var(--ink)}
.dsector{font-size:11px;font-family:var(--mono);color:var(--c,var(--muted))}
.dname{font-size:21px;font-weight:800;letter-spacing:-.02em;margin:2px 0 0}
.dwhere{font-size:12px;color:var(--dim);margin-top:2px}
.dprice{display:flex;align-items:baseline;gap:10px;margin-top:9px}
.dprice .p{font-family:var(--mono);font-size:27px;font-weight:600}
.dtabs{display:flex;gap:2px;padding:9px 12px 0;border-bottom:1px solid var(--line);overflow-x:auto}
.dtabs button{padding:8px 11px;font-size:12.5px;color:var(--muted);border-bottom:2px solid transparent;
  margin-bottom:-1px;white-space:nowrap}
.dtabs button[aria-selected=true]{color:var(--ink);border-bottom-color:var(--c,var(--focus))}
.dbody{flex:1;overflow-y:auto;padding:14px 16px calc(24px + env(safe-area-inset-bottom,0px));scrollbar-width:thin}
.dbody::-webkit-scrollbar{width:7px} .dbody::-webkit-scrollbar-thumb{background:var(--line);border-radius:4px}
.kpis{display:grid;grid-template-columns:repeat(3,1fr);gap:8px;margin-bottom:13px}
.kpi{border:1px solid var(--line);border-radius:9px;padding:8px 10px;background:rgba(5,9,20,.6)}
.kpi .k{font-size:10.5px;color:var(--dim)}
.kpi .v{font-family:var(--mono);font-size:15.5px;margin-top:3px}
.range{margin:2px 0 14px}
.rangebar{height:6px;border-radius:3px;background:linear-gradient(90deg,rgba(255,95,109,.5),rgba(61,220,151,.5));position:relative}
.rangebar i{position:absolute;top:-4px;width:2px;height:14px;background:var(--ink);border-radius:1px}
.rangelabels{display:flex;justify-content:space-between;font-family:var(--mono);font-size:11px;color:var(--dim);margin-top:5px}
.chartwrap{border:1px solid var(--line);border-radius:10px;background:rgba(5,9,20,.55);padding:8px;margin-bottom:12px}
.chartwrap h4{margin:2px 4px 6px;font-size:12px;font-weight:500;color:var(--muted)}
.chart{width:100%;height:210px} .chart.tall{height:262px}
.toggles{display:flex;gap:6px;flex-wrap:wrap;margin-bottom:10px}
.toggles button{padding:4px 10px;border:1px solid var(--line);border-radius:999px;font-size:11px;
  font-family:var(--mono);color:var(--dim)}
.toggles button[aria-pressed=true]{color:var(--ink);border-color:var(--line2);background:rgba(122,152,204,.12)}
.prose{font-size:13px;color:var(--muted);line-height:1.6}
.deflist{display:grid;grid-template-columns:auto 1fr;gap:6px 14px;font-size:12.5px;margin:12px 0}
.deflist dt{color:var(--dim)} .deflist dd{margin:0;font-family:var(--mono)}
.note{font-size:11px;color:var(--dim);font-family:var(--mono);margin:-4px 0 10px}

/* ---------------- board ---------------- */
.board{position:fixed;z-index:4;inset:var(--boardtop,150px) 0 0;overflow-y:auto;padding:8px 16px 40px;
  background:linear-gradient(180deg,rgba(3,5,11,.92),rgba(3,5,11,.975) 140px);display:none}
.board.show{display:block}
body.board-view .foot{display:none}
@media (min-width:1181px){
  body.board-view .rail, body.board-view .mapctl{display:none}
}
.grid{display:grid;grid-template-columns:repeat(12,1fr);gap:13px;max-width:1720px;margin:0 auto}
.panel{border:1px solid var(--line);border-radius:12px;background:rgba(7,12,26,.75);padding:12px 14px}
.panel h3{margin:0 0 2px;font-size:13px;font-weight:600}
.panel p.hint{margin:0 0 9px;font-size:11.5px;color:var(--dim)}
.span4{grid-column:span 4} .span5{grid-column:span 5} .span7{grid-column:span 7}
.span8{grid-column:span 8} .span12{grid-column:span 12}
.tablewrap{max-height:430px;overflow:auto;scrollbar-width:thin}
table{width:100%;border-collapse:collapse;font-size:12.5px}
th{position:sticky;top:0;z-index:1;background:#070C1A;text-align:right;padding:7px 8px;color:var(--dim);
  font-weight:500;font-size:11px;cursor:pointer;border-bottom:1px solid var(--line);white-space:nowrap}
th:first-child,td:first-child{text-align:left}
th[data-active]{color:var(--focus)}
td{padding:6px 8px;text-align:right;border-bottom:1px solid rgba(122,152,204,.08);font-family:var(--mono);white-space:nowrap}
td.name{font-family:var(--sans);max-width:260px;overflow:hidden;text-overflow:ellipsis}
tbody tr{cursor:pointer}
tbody tr:hover td{background:rgba(122,152,204,.08)}
.badge{display:inline-block;width:7px;height:7px;border-radius:50%;margin-right:7px;vertical-align:middle;background:var(--c)}

/* ---------------- labels on the map ---------------- */
.lbl{font-family:var(--mono);font-size:10.5px;color:#E7EEFB;text-shadow:0 1px 5px #000,0 0 2px #000;
  white-space:nowrap;pointer-events:none;transform:translate(10px,-2px);opacity:.9}
.foot{position:fixed;z-index:6;left:16px;bottom:12px;font-size:10px;color:var(--dim);font-family:var(--mono);
  max-width:46vw}
.toast{position:fixed;z-index:9;left:50%;bottom:26px;transform:translate(-50%,20px);opacity:0;
  background:var(--panel2);border:1px solid var(--line2);border-radius:8px;padding:8px 14px;font-size:12px;
  transition:.25s;pointer-events:none}
.toast.show{opacity:1;transform:translate(-50%,0)}
@media (prefers-reduced-motion:reduce){*{transition:none!important}}

/* ---------------- the dock: three fixed panels on desktop, one sheet on mobile ---------------- */
#dock{display:contents}
.sheet-handle{display:none}
@media (max-width:1180px){
  .span4,.span5,.span7,.span8{grid-column:span 12}
  .foot{display:none}
  #dock{display:flex;flex-direction:column;gap:9px;position:fixed;z-index:7;left:0;right:0;bottom:0;
    height:min(76vh,620px);background:var(--panel2);backdrop-filter:blur(14px);
    border-top:1px solid var(--line2);border-radius:16px 16px 0 0;box-shadow:0 -14px 40px rgba(0,0,0,.55);
    padding:0 12px calc(12px + env(safe-area-inset-bottom,0px));overflow:hidden;
    transform:translateY(calc(100% - 52px - env(safe-area-inset-bottom,0px)));
    transition:transform .3s cubic-bezier(.22,.8,.3,1)}
  #dock.open{transform:none}
  #dock .sectors,#dock .rail,#dock .mapctl{position:static;inset:auto;width:auto;max-width:none;
    display:flex;flex-wrap:wrap;gap:6px}
  #dock .rail{flex:1 1 auto;min-height:0;flex-direction:column;flex-wrap:nowrap;overflow-y:auto;gap:7px;
    padding-right:2px}
  #dock .mapctl{flex-direction:row;align-items:center}
  .sheet-handle{display:flex;align-items:center;gap:10px;width:100%;padding:14px 4px 6px;
    font-size:13px;color:var(--ink);flex:0 0 auto}
  .sheet-handle .grab{width:38px;height:4px;border-radius:2px;background:var(--line2);flex:0 0 auto}
  .sheet-handle .caret{margin-left:auto;color:var(--dim);transition:transform .3s}
  #dock.open .sheet-handle .caret{transform:rotate(180deg)}
  .board{padding-bottom:84px}
}
@media (max-width:760px){
  .hud{gap:8px;padding:calc(8px + env(safe-area-inset-top,0px)) 10px 8px}
  .brand b{font-size:17px} .brand span{display:none}
  .tabs button{padding:6px 13px}
  .search{order:5;flex:1 1 100%;min-width:0}
  .slicers{gap:7px;width:100%;justify-content:flex-start}
  .seg button{padding:5px 9px}
  .drawer{width:100vw}
  .kpis{grid-template-columns:repeat(2,1fr)}
}
</style>
</head>
<body>
<canvas id="galaxy"></canvas>
<div id="globe"></div>
<div class="vignette"></div>

<header class="hud">
  <div class="brand"><b>Ticker Orbit</b><span id="stamp"></span></div>
  <div class="tabs" role="tablist">
    <button role="tab" id="tab-globe" aria-selected="true">Globe</button>
    <button role="tab" id="tab-board" aria-selected="false">Board</button>
  </div>
  <label class="search"><span style="color:var(--dim);font-size:12px">⌕</span>
    <input id="q" type="search" placeholder="ticker, company, city" autocomplete="off"></label>
  <div class="slicers">
    <div class="seg" id="periods"></div>
    <button class="ghost" id="tour">Tour</button>
    <button class="ghost" id="fs">Full screen</button>
  </div>
</header>
<div id="dock">
  <button class="sheet-handle" id="dockToggle" aria-expanded="false">
    <span class="grab"></span><span id="dockLabel">Companies</span><span class="caret">▴</span>
  </button>
  <div class="sectors" id="chips"></div>
  <nav class="rail" id="rail" aria-label="Companies"></nav>
  <div class="mapctl">
    <div class="row" id="basemaps"></div>
    <div class="row">
      <button id="t-terrain" aria-pressed="true">3D terrain</button>
      <button id="t-labels" aria-pressed="true">Labels</button>
      <button id="t-spin" aria-pressed="true">Spin</button>
    <button id="t-reset">Reset view</button>
    </div>
  </div>
</div>

<section class="board" id="board">
  <div class="grid">
    <div class="panel span12">
      <h3>Leaderboard</h3>
      <p class="hint" id="leadhint"></p>
      <div class="tablewrap"><table id="lead"><thead></thead><tbody></tbody></table></div>
    </div>
    <div class="panel span7">
      <h3>Risk against reward</h3>
      <p class="hint">Annualized volatility versus annualized return. Bubble area is market cap.</p>
      <div class="chart tall" id="c-scatter"></div>
    </div>
    <div class="panel span5">
      <h3>How they move together</h3>
      <p class="hint" id="corrhint">Correlation of returns across the top of the current sort.</p>
      <div class="chart tall" id="c-corr"></div>
    </div>
    <div class="panel span8">
      <h3>Equal-weight basket of the current selection</h3>
      <p class="hint" id="porthint"></p>
      <div class="chart" id="c-port"></div>
    </div>
    <div class="panel span4">
      <h3>Weight by sector</h3>
      <p class="hint">Market cap of the current selection, grouped by sector.</p>
      <div class="chart" id="c-cap"></div>
    </div>
  </div>
</section>

<aside class="drawer" id="drawer" aria-label="Company detail">
  <div class="dhead" id="dhead"></div>
  <div class="dtabs" id="dtabs"></div>
  <div class="dbody" id="dbody"></div>
</aside>

<div class="foot" id="foot"></div>
<div class="toast" id="toast"></div>

<script src="https://cdn.jsdelivr.net/npm/maplibre-gl@5.24.0/dist/maplibre-gl.js"></script>
<script src="https://cdn.jsdelivr.net/npm/echarts@5.6.0/dist/echarts.min.js"></script>
<script>
const DATA = /*__PAYLOAD__*/;

/* ============================ helpers ============================ */
const $ = s => document.querySelector(s);
const el = (t, cls, html) => { const n = document.createElement(t); if (cls) n.className = cls; if (html != null) n.innerHTML = html; return n; };
const PALETTE = {
  'Information Technology':'#4FD3E8', 'Health Care':'#7CE7B0', 'Financials':'#E8B14C',
  'Consumer Discretionary':'#FF8FA3', 'Communication Services':'#A78BFA', 'Industrials':'#F2884B',
  'Consumer Staples':'#C9D66B', 'Energy':'#FF6B57', 'Utilities':'#6FA8FF',
  'Real Estate':'#D98FD9', 'Materials':'#8FBCA8', 'Insurance':'#4FD3E8', 'Banking':'#E8B14C',
  'Asset management':'#FF8FA3', 'Pharmaceuticals':'#A78BFA', 'Benchmark':'#9FB3D1'
};
const color = c => PALETTE[c.sector] || '#9FB3D1';
const pct = (v,d=1) => (v==null||!isFinite(v)) ? '—' : (v*100).toFixed(d)+'%';
const spct = (v,d=1) => (v==null||!isFinite(v)) ? '—' : (v>=0?'+':'')+(v*100).toFixed(d)+'%';
const usd = (v,d=2) => (v==null||!isFinite(v)) ? '—' : '$'+v.toLocaleString('en-US',{minimumFractionDigits:d,maximumFractionDigits:d});
const num = (v,d=2) => (v==null||!isFinite(v)) ? '—' : v.toFixed(d);
const cap = v => { if(!v||!isFinite(v)) return '—'; const u=[['T',1e12],['B',1e9],['M',1e6]];
  for(const [s,n] of u) if(v>=n) return '$'+(v/n).toFixed(v/n>=100?0:1)+s; return '$'+v.toFixed(0); };
const vlm = v => (!v||!isFinite(v)) ? '—' : v>=1e6 ? (v/1e6).toFixed(1)+'M' : (v/1e3).toFixed(0)+'K';
const cls = v => v==null||!isFinite(v) ? '' : (v>=0?'up':'down');
function toast(m){ const t=$('#toast'); t.textContent=m; t.classList.add('show'); clearTimeout(t._h); t._h=setTimeout(()=>t.classList.remove('show'),2400); }

/* ============================ math ============================ */
const mean = a => a.length ? a.reduce((s,v)=>s+v,0)/a.length : NaN;
function sd(a){ if(a.length<2) return NaN; const m=mean(a); return Math.sqrt(a.reduce((s,v)=>s+(v-m)*(v-m),0)/(a.length-1)); }
function retsOf(a){ const r=[]; for(let i=1;i<a.length;i++) r.push(a[i]/a[i-1]-1); return r; }
function maxDrawdown(p){ let peak=p[0], dd=0; for(const v of p){ if(v>peak) peak=v; const d=v/peak-1; if(d<dd) dd=d; } return dd; }
function corr(a,b){ const n=Math.min(a.length,b.length); if(n<3) return NaN;
  const x=a.slice(-n), y=b.slice(-n), mx=mean(x), my=mean(y); let s=0,dx=0,dy=0;
  for(let i=0;i<n;i++){ const p=x[i]-mx, q=y[i]-my; s+=p*q; dx+=p*p; dy+=q*q; } return s/Math.sqrt(dx*dy); }
function cumulative(rs){ const o=[1]; let v=1; for(const r of rs){ v*=(1+r); o.push(v); } return o; }
function sma(a,n){ const o=Array(a.length).fill(null); let s=0;
  for(let i=0;i<a.length;i++){ s+=a[i]; if(i>=n) s-=a[i-n]; if(i>=n-1) o[i]=s/n; } return o; }
function ema(a,n){ const o=Array(a.length).fill(null); const k=2/(n+1); let p=null;
  for(let i=0;i<a.length;i++){ p = p==null ? a[i] : a[i]*k+p*(1-k); o[i]=p; } return o; }
function bollinger(a,n=20,m=2){ const mid=sma(a,n), up=[], lo=[];
  for(let i=0;i<a.length;i++){ if(mid[i]==null){ up.push(null); lo.push(null); continue; }
    const s=sd(a.slice(i-n+1,i+1)); up.push(mid[i]+m*s); lo.push(mid[i]-m*s); } return {mid,up,lo}; }
function rsi(a,n=14){ const o=Array(a.length).fill(null); let g=0,l=0;
  for(let i=1;i<a.length;i++){ const d=a[i]-a[i-1], gg=Math.max(d,0), ll=Math.max(-d,0);
    if(i<=n){ g+=gg/n; l+=ll/n; if(i===n) o[i]=100-100/(1+g/(l||1e-9)); }
    else { g=(g*(n-1)+gg)/n; l=(l*(n-1)+ll)/n; o[i]=100-100/(1+g/(l||1e-9)); } } return o; }
function macd(a){ const f=ema(a,12), s=ema(a,26), line=a.map((_,i)=>f[i]-s[i]), sig=ema(line,9);
  return {line, sig, hist: line.map((v,i)=>v-sig[i])}; }

/* ============================ series & windows ============================ */
const COMPANIES = DATA.companies.filter(c => c.ticker !== DATA.benchmark);
const BENCH = DATA.companies.find(c => c.ticker === DATA.benchmark);
const BY_T = Object.fromEntries(DATA.companies.map(c => [c.ticker, c]));
const SECTORS = [...new Set(COMPANIES.map(c => c.sector))].sort();
const PERIODS = ['1M','6M','YTD','1Y','3Y','5Y'];
const FRACTION = { '1M':1/12, '6M':0.5, '1Y':1, '3Y':3 };
const state = { period:'1Y', sectors:new Set(SECTORS), active:null, view:'globe', dtab:'price',
                overlays:new Set(['sma']), query:'', sortKey:'total', sortDir:-1, rows:200 };

function seriesOf(c, prefer){
  const daily = prefer !== 'w' && c.d;
  const s = daily ? c.d : c.w;
  return { dates: daily ? DATA.datesD : DATA.datesW, close: s.close, volume: s.volume,
           ppy: daily ? 252 : 52, tag: daily ? 'daily' : 'weekly' };
}
function startIndex(S){
  const n = S.dates.length;
  if (state.period === '5Y') return 0;
  if (state.period === 'YTD'){ const y = S.dates[n-1].slice(0,4);
    const i = S.dates.findIndex(d => d.slice(0,4) === y); return Math.max(0, i-1); }
  return Math.max(0, n - 1 - Math.round(FRACTION[state.period] * S.ppy));
}
const _cache = new Map();
function win(c, prefer){
  const S = seriesOf(c, prefer);
  const key = c.ticker + '|' + state.period + '|' + S.tag;
  if (_cache.has(key)) return _cache.get(key);
  let first = 0; while (first < S.close.length && S.close[first] == null) first++;
  const i0 = Math.max(first, startIndex(S));
  const close = S.close.slice(i0), dates = S.dates.slice(i0);
  const B = seriesOf(BENCH, S.tag === 'daily' ? 'd' : 'w');
  const r = retsOf(close), br = retsOf(B.close.slice(i0));
  const annRet = mean(r) * S.ppy, annVol = sd(r) * Math.sqrt(S.ppy);
  const co = corr(r, br), down = r.filter(v => v < 0);
  const o = { i0, dates, close, rets:r, benchRets:br, ppy:S.ppy, tag:S.tag,
    volume: S.volume.slice(i0),
    total: close.length > 1 ? close[close.length-1]/close[0] - 1 : NaN,
    annRet, annVol, sharpe:(annRet - DATA.riskFree)/annVol,
    sortino:(annRet - DATA.riskFree)/(sd(down)*Math.sqrt(S.ppy)),
    maxDD: maxDrawdown(close), corrBench: co, beta: co * (sd(r)/sd(br)),
    last: close[close.length-1], hi: Math.max(...close), lo: Math.min(...close) };
  o.alpha = o.annRet - (DATA.riskFree + o.beta * (mean(br)*S.ppy - DATA.riskFree));
  _cache.set(key, o);
  return o;
}
function matches(c){
  if (!state.sectors.has(c.sector)) return false;
  const q = state.query.trim().toLowerCase();
  if (!q) return true;
  return (c.ticker + ' ' + c.name + ' ' + (c.hq ? c.hq.city : '') + ' ' + (c.industry||'')).toLowerCase().includes(q);
}
const shown = () => COMPANIES.filter(matches);

/* ============================ galaxy ============================ */
let camX = 0, camY = 0;      // accumulated parallax offset, in device pixels
(function galaxy(){
  const cv = $('#galaxy'), ctx = cv.getContext('2d');
  const reduce = matchMedia('(prefers-reduced-motion: reduce)').matches;
  let w, h, dpr, stars = [], clouds = [], shoot = null, last = null;
  const PX_LNG = 7, PX_LAT = 9, PX_BEARING = 5, PX_PITCH = 3;
  const wrap = (v, m) => ((v % m) + m) % m;

  function band(x){ return h * (0.42 + 0.22 * Math.sin((x / w) * Math.PI * 1.4 - 0.6)); }
  function seed(){
    dpr = Math.min(devicePixelRatio || 1, 2);
    w = cv.width = innerWidth * dpr; h = cv.height = innerHeight * dpr;
    cv.style.width = innerWidth + 'px'; cv.style.height = innerHeight + 'px';
    stars = Array.from({ length: Math.round(innerWidth * innerHeight / 2600) }, () => {
      const x = Math.random() * w;
      const inBand = Math.random() < 0.45;
      const y = inBand ? band(x) + (Math.random() - 0.5) * h * 0.22 : Math.random() * h;
      return { x, y, z: Math.random() ** 2 * 0.85 + 0.15, t: Math.random() * 6.28 };
    });
    clouds = Array.from({ length: 6 }, (_, i) => {
      const x = (i + 0.5) / 6 * w + (Math.random() - 0.5) * w * 0.1;
      return { x, y: band(x) + (Math.random() - 0.5) * h * 0.14,
               r: (0.16 + Math.random() * 0.2) * Math.min(w, h),
               c: ['rgba(70,110,220,', 'rgba(150,90,220,', 'rgba(40,150,190,'][i % 3] };
    });
  }

  /* the camera drives the field: spin, drag, flyTo and the tour all land here */
  function camera(){
    if (typeof map === 'undefined' || !map.getCenter) return;
    const c = map.getCenter(), b = map.getBearing(), p = map.getPitch(), z = Math.max(0.5, map.getZoom());
    if (last){
      let dLng = c.lng - last.lng;
      if (dLng > 180) dLng -= 360; else if (dLng < -180) dLng += 360;
      let dBearing = b - last.bearing;
      if (dBearing > 180) dBearing -= 360; else if (dBearing < -180) dBearing += 360;
      const depth = dpr * Math.min(1, 2 / z);     // parallax eases off as you dive toward the ground
      camX -= (dLng * PX_LNG + dBearing * PX_BEARING) * depth;
      camY += ((c.lat - last.lat) * PX_LAT + (p - last.pitch) * PX_PITCH) * depth;
    }
    last = { lng:c.lng, lat:c.lat, bearing:b, pitch:p };
  }

  function draw(ts){
    camera();
    if (!reduce) camX -= 0.02 * dpr;              // a whisper of drift when nothing is moving
    ctx.fillStyle = '#03050B'; ctx.fillRect(0, 0, w, h);

    for (const c of clouds){
      const cx = wrap(c.x + camX * 0.28, w), cy = wrap(c.y + camY * 0.28, h);
      for (const ox of [-w, 0, w]){
        const x = cx + ox;
        if (x + c.r < 0 || x - c.r > w) continue;
        const g = ctx.createRadialGradient(x, cy, 0, x, cy, c.r);
        g.addColorStop(0, c.c + '0.10)'); g.addColorStop(0.55, c.c + '0.04)'); g.addColorStop(1, c.c + '0)');
        ctx.fillStyle = g; ctx.beginPath(); ctx.arc(x, cy, c.r, 0, 6.29); ctx.fill();
      }
    }

    for (const p of stars){
      const tw = reduce ? 0.75 : 0.55 + 0.45 * Math.sin(ts / 900 + p.t);
      ctx.globalAlpha = p.z * tw;
      ctx.fillStyle = p.z > 0.8 ? '#CFE0FF' : p.z > 0.45 ? '#EAF0FC' : '#93A9CC';
      const s = p.z * 1.9 * dpr;
      ctx.fillRect(wrap(p.x + camX * p.z, w), wrap(p.y + camY * p.z, h), s, s);
    }
    ctx.globalAlpha = 1;

    if (!reduce){
      if (!shoot && Math.random() < 0.0016) shoot = { x: Math.random()*w*0.7, y: Math.random()*h*0.5, life: 1 };
      if (shoot){
        const len = 140 * dpr;
        const g = ctx.createLinearGradient(shoot.x, shoot.y, shoot.x + len, shoot.y + len*0.35);
        g.addColorStop(0, 'rgba(255,255,255,' + shoot.life * 0.9 + ')');
        g.addColorStop(1, 'rgba(255,255,255,0)');
        ctx.strokeStyle = g; ctx.lineWidth = 1.6 * dpr; ctx.beginPath();
        ctx.moveTo(shoot.x, shoot.y); ctx.lineTo(shoot.x + len, shoot.y + len*0.35); ctx.stroke();
        shoot.x += 9 * dpr; shoot.y += 3.2 * dpr; shoot.life -= 0.02;
        if (shoot.life <= 0) shoot = null;
      }
    }
    requestAnimationFrame(draw);
  }
  seed(); addEventListener('resize', seed); requestAnimationFrame(draw);
})();

/* ============================ globe ============================ */
const ESRI = 'https://server.arcgisonline.com/ArcGIS/rest/services/';
const BASEMAPS = {
  satellite: { label:'Satellite', tiles:[ESRI + 'World_Imagery/MapServer/tile/{z}/{y}/{x}'],
               attribution:'Imagery © Esri, Maxar, Earthstar Geographics' },
  dark:      { label:'Dark', tiles:[ESRI + 'Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}'],
               attribution:'© Esri, HERE, Garmin, © OpenStreetMap contributors' },
  terrain:   { label:'Terrain', tiles:[ESRI + 'World_Topo_Map/MapServer/tile/{z}/{y}/{x}'],
               attribution:'© Esri, USGS, NOAA' }
};
const style = {
  version: 8,
  projection: { type: 'globe' },
  sources: {
    sat:    { type:'raster', tileSize:256, maxzoom:19, tiles:BASEMAPS.satellite.tiles, attribution:BASEMAPS.satellite.attribution },
    dark:   { type:'raster', tileSize:256, maxzoom:16, tiles:BASEMAPS.dark.tiles, attribution:BASEMAPS.dark.attribution },
    topo:   { type:'raster', tileSize:256, maxzoom:19, tiles:BASEMAPS.terrain.tiles, attribution:BASEMAPS.terrain.attribution },
    places: { type:'raster', tileSize:256, maxzoom:19,
              tiles:[ESRI + 'Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}'] },
    dem:    { type:'raster-dem', encoding:'terrarium', tileSize:256, maxzoom:14,
              tiles:['https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{z}/{x}/{y}.png'],
              attribution:'Elevation: AWS Terrain Tiles, SRTM / GMTED' },
    hq:     { type:'geojson', data:{ type:'FeatureCollection', features:[] },
              cluster:true, clusterRadius:42, clusterMaxZoom:6, clusterProperties:{ } }
  },
  layers: [
    { id:'sat',  type:'raster', source:'sat',  layout:{ visibility:'visible' } },
    { id:'dark', type:'raster', source:'dark', layout:{ visibility:'none' } },
    { id:'topo', type:'raster', source:'topo', layout:{ visibility:'none' } },
    { id:'hillshade', type:'hillshade', source:'dem',
      paint:{ 'hillshade-exaggeration':0.35, 'hillshade-shadow-color':'#000814', 'hillshade-highlight-color':'#8FB4E8' } },
    { id:'places', type:'raster', source:'places', paint:{ 'raster-opacity':0.85 }, layout:{ visibility:'visible' } },
    { id:'clusters', type:'circle', source:'hq', filter:['has','point_count'],
      paint:{ 'circle-color':'rgba(79,211,232,0.28)', 'circle-stroke-color':'#4FD3E8', 'circle-stroke-width':1.4,
              'circle-radius':['interpolate',['linear'],['get','point_count'],2,13,20,22,120,34] } },
    { id:'cluster-count', type:'circle', source:'hq', filter:['has','point_count'],
      paint:{ 'circle-radius':2.5, 'circle-color':'#E7EEFB' } },

    { id:'halo', type:'circle', source:'hq', filter:['!',['has','point_count']],
      paint:{ 'circle-color':['get','color'], 'circle-opacity':0.18, 'circle-blur':0.6,
              'circle-radius':['interpolate',['linear'],['zoom'],1,9,4,14,10,24] } },
    { id:'dots', type:'circle', source:'hq', filter:['!',['has','point_count']],
      paint:{ 'circle-color':['get','color'],
              'circle-radius':['interpolate',['linear'],['zoom'],1,4.5,4,6.5,10,11],
              'circle-stroke-color':'#03050B', 'circle-stroke-width':1,
              'circle-opacity':0.95, 'circle-blur':0.15 } }
  ],
  sky: { 'sky-color':'rgba(4,10,28,0)', 'horizon-color':'rgba(86,160,255,0.55)', 'fog-color':'rgba(4,8,20,0)',
         'sky-horizon-blend':0.6, 'horizon-fog-blend':0.5, 'fog-ground-blend':0.2,
         'atmosphere-blend':['interpolate',['linear'],['zoom'],0,0.85,5,0.35,9,0] }
};
const map = new maplibregl.Map({
  container:'globe', style, center:[-84,32], zoom:1.75, pitch:0,
  minZoom:0.85, maxZoom:15, maxPitch:66, bearingSnap:12,
  attributionControl:{ compact:true }, antialias:true
});
// two-finger twists were the main way the globe ended up upside down
map.touchZoomRotate.disableRotation();
map.addControl(new maplibregl.NavigationControl({ visualizePitch:true }), 'bottom-right');
map.addControl(new maplibregl.FullscreenControl({ container: document.documentElement }), 'bottom-right');

let terrainOn = true, spinning = true, tourTimer = null;
map.on('load', () => {
  try { map.setProjection({ type:'globe' }); } catch(e){}
  try { map.setTerrain({ source:'dem', exaggeration:1.45 }); } catch(e){ terrainOn = false; }
  refreshPoints();
  requestAnimationFrame(spin);
});
map.on('error', e => { /* a missing tile should never take the page down */ });
map.on('moveend', () => {
  const z = map.getZoom(), p = map.getPitch(), c = map.getCenter();
  if (!isFinite(z) || !isFinite(c.lat) || !isFinite(c.lng) || Math.abs(c.lat) > 89){
    map.jumpTo({ center:[c.lng || 0, Math.max(-85, Math.min(85, c.lat || 0))], zoom:1.9, pitch:0, bearing:0 });
  } else if (z > 15 || p > 66){
    map.easeTo({ zoom:Math.min(z, 15), pitch:Math.min(p, 66), duration:400 });
  }
});

function features(){
  return DATA.companies.filter(c => c.hq && (c.ticker === DATA.benchmark || matches(c))).map(c => ({
    type:'Feature', properties:{ ticker:c.ticker, color:color(c) },
    geometry:{ type:'Point', coordinates:[c.hq.lon, c.hq.lat] }
  }));
}
function refreshPoints(){
  const src = map.getSource('hq');
  if (src) src.setData({ type:'FeatureCollection', features: features() });
  refreshLabels();
}

/* labels for whatever is actually in view, capped so 500 markers never hit the DOM */
const labelMarkers = new Map();
function refreshLabels(){
  if (!map.isStyleLoaded || !map.getSource('hq')) return;
  const z = map.getZoom(), b = map.getBounds();
  const keep = new Set();
  if (z >= 3.6){
    const visible = DATA.companies
      .filter(c => c.hq && (c.ticker === DATA.benchmark || matches(c)))
      .filter(c => b.contains([c.hq.lon, c.hq.lat]))
      .sort((a,x) => (x.profile.marketCap||0) - (a.profile.marketCap||0))
      .slice(0, 28);
    for (const c of visible){
      keep.add(c.ticker);
      if (!labelMarkers.has(c.ticker)){
        const n = el('div','lbl', c.ticker);
        const m = new maplibregl.Marker({ element:n, anchor:'left' }).setLngLat([c.hq.lon, c.hq.lat]).addTo(map);
        labelMarkers.set(c.ticker, m);
      }
    }
  }
  for (const [t, m] of labelMarkers) if (!keep.has(t)){ m.remove(); labelMarkers.delete(t); }
}
map.on('moveend', refreshLabels);
map.on('zoomend', refreshLabels);

const popup = new maplibregl.Popup({ closeButton:false, className:'popup', offset:12 });
map.on('mouseenter','dots', e => {
  map.getCanvas().style.cursor = 'pointer';
  const c = BY_T[e.features[0].properties.ticker]; if (!c) return;
  const w = win(c, 'w');
  popup.setLngLat(e.features[0].geometry.coordinates)
    .setHTML('<b>' + c.ticker + '</b> ' + c.name + '<br>' + (c.hq ? c.hq.city : '') +
             '<br>' + usd(w.last) + ' · <span class="' + cls(w.total) + '">' + spct(w.total) + '</span> ' + state.period)
    .addTo(map);
});
map.on('mouseleave','dots', () => { map.getCanvas().style.cursor = ''; popup.remove(); });
map.on('click','dots', e => select(e.features[0].properties.ticker, true));
map.on('click','clusters', e => {
  const f = map.queryRenderedFeatures(e.point, { layers:['clusters'] })[0];
  map.getSource('hq').getClusterExpansionZoom(f.properties.cluster_id).then(z => {
    spinning = false;
    map.easeTo({ center:f.geometry.coordinates, zoom:Math.min(z + 0.4, 9), duration:900 });
  }).catch(()=>{});
});
map.on('mouseenter','clusters', () => map.getCanvas().style.cursor = 'pointer');
map.on('mouseleave','clusters', () => map.getCanvas().style.cursor = '');

function spin(){
  requestAnimationFrame(spin);
  if (!spinning || map.isMoving() || document.hidden || map.getZoom() > 4) return;
  const c = map.getCenter(); map.setCenter([c.lng + 0.04, c.lat]);
}
['mousedown','touchstart','wheel'].forEach(ev =>
  map.getCanvas().addEventListener(ev, () => { setSpin(false); stopTour(); }, { passive:true }));

function flyToHQ(c){
  if (!c || !c.hq) return;
  setSpin(false);
  const lean = ((Math.abs(Math.round(c.hq.lon * 97)) % 40) - 20);   // -20..20 degrees, deterministic
  map.flyTo({ center:[c.hq.lon, c.hq.lat], zoom:12, pitch:54,
              bearing:lean, duration:3000, curve:1.45, essential:true });
}
function setSpin(on){ spinning = on; $('#t-spin').setAttribute('aria-pressed', on);
  if (on) map.easeTo({ pitch:0, bearing:0, zoom:Math.min(map.getZoom(), 2.2), duration:1400 }); }
function stopTour(){ if (tourTimer){ clearInterval(tourTimer); tourTimer = null; $('#tour').setAttribute('aria-pressed','false'); } }

/* basemap + layer controls */
const bm = $('#basemaps');
Object.entries(BASEMAPS).forEach(([k, v], i) => {
  const b = el('button', null, v.label);
  b.setAttribute('aria-pressed', i === 0);
  b.onclick = () => {
    Object.keys(BASEMAPS).forEach(id => map.setLayoutProperty(id === 'satellite' ? 'sat' : id === 'dark' ? 'dark' : 'topo',
      'visibility', id === k ? 'visible' : 'none'));
    [...bm.children].forEach(x => x.setAttribute('aria-pressed', x === b));
  };
  bm.appendChild(b);
});
$('#t-terrain').onclick = () => {
  terrainOn = !terrainOn;
  map.setTerrain(terrainOn ? { source:'dem', exaggeration:1.45 } : null);
  map.setPaintProperty('hillshade','hillshade-exaggeration', terrainOn ? 0.35 : 0);
  $('#t-terrain').setAttribute('aria-pressed', terrainOn);
};
$('#t-labels').onclick = () => {
  const on = map.getLayoutProperty('places','visibility') !== 'none';
  map.setLayoutProperty('places','visibility', on ? 'none' : 'visible');
  $('#t-labels').setAttribute('aria-pressed', !on);
};
$('#t-spin').onclick = () => setSpin(!spinning);
$('#t-reset').onclick = () => {
  stopTour();
  map.easeTo({ center:map.getCenter(), zoom:1.9, pitch:0, bearing:0, duration:1400, essential:true });
};
$('#fs').onclick = () => {
  if (document.fullscreenElement) document.exitFullscreen();
  else (document.documentElement.requestFullscreen ? document.documentElement.requestFullscreen()
        : document.documentElement.webkitRequestFullscreen()).catch?.(() => toast('Full screen was blocked here — open the HTML file directly'));
};
addEventListener('fullscreenchange', () => {
  $('#fs').setAttribute('aria-pressed', !!document.fullscreenElement);
  $('#fs').textContent = document.fullscreenElement ? 'Exit full screen' : 'Full screen';
  setTimeout(() => { map.resize(); charts.forEach(c => !c.isDisposed() && c.resize()); }, 250);
});

/* ============================ HUD ============================ */
$('#stamp').textContent = DATA.datesW[DATA.datesW.length-1] + ' · ' + COMPANIES.length + ' companies';
$('#foot').innerHTML = 'Prices: Yahoo Finance via yfinance · basemaps © Esri and contributors · elevation: AWS Terrain Tiles · ' +
  'built ' + DATA.generated + ' · educational use, not investment advice';

const chips = $('#chips');
SECTORS.forEach(s => {
  const b = el('button','chip','<span class="dot"></span>' + s);
  b.style.setProperty('--c', PALETTE[s] || '#9FB3D1');
  b.setAttribute('aria-pressed','true');
  b.onclick = () => {
    state.sectors.has(s) ? state.sectors.delete(s) : state.sectors.add(s);
    if (!state.sectors.size) state.sectors.add(s);
    b.setAttribute('aria-pressed', state.sectors.has(s));
    renderAll();
  };
  chips.appendChild(b);
});
const periods = $('#periods');
PERIODS.forEach(p => {
  const b = el('button', null, p);
  b.setAttribute('aria-pressed', p === state.period);
  b.onclick = () => { state.period = p; [...periods.children].forEach(x => x.setAttribute('aria-pressed', x.textContent === p)); renderAll(); };
  periods.appendChild(b);
});
let qTimer;
$('#q').addEventListener('input', e => {
  clearTimeout(qTimer);
  qTimer = setTimeout(() => { state.query = e.target.value; renderAll(); }, 180);
});
$('#tab-globe').onclick = () => setView('globe');
$('#tab-board').onclick = () => setView('board');
const MOBILE = () => matchMedia('(max-width:1180px)').matches;
function sizeBoard(){
  const hud = document.querySelector('.hud').getBoundingClientRect().bottom;
  const chips = $('#chips');
  const bottom = MOBILE() ? hud : Math.max(hud, chips.getBoundingClientRect().bottom);
  document.documentElement.style.setProperty('--boardtop', Math.round(bottom + 10) + 'px');
}
const dock = document.getElementById('dock');
$('#dockToggle').onclick = () => {
  const open = dock.classList.toggle('open');
  $('#dockToggle').setAttribute('aria-expanded', open);
};
function setView(v){
  state.view = v;
  $('#tab-globe').setAttribute('aria-selected', v === 'globe');
  $('#tab-board').setAttribute('aria-selected', v === 'board');
  $('#board').classList.toggle('show', v === 'board');
  document.body.classList.toggle('board-view', v === 'board');
  if (v === 'board'){ stopTour(); sizeBoard(); renderBoard(); }
}
addEventListener('resize', () => { if (state.view === 'board') sizeBoard(); });
$('#tour').onclick = () => {
  if (tourTimer){ stopTour(); return; }
  const list = shown().slice().sort((a,b) => (b.profile.marketCap||0) - (a.profile.marketCap||0)).slice(0, 40);
  if (!list.length) return;
  let i = 0;
  $('#tour').setAttribute('aria-pressed','true');
  setView('globe');
  const hop = () => select(list[i++ % list.length].ticker, true);
  hop(); tourTimer = setInterval(hop, 8000);
};

/* ============================ roster ============================ */
const RAIL_MAX = 60;
function sparkline(vals, stroke){
  const n = vals.length, step = Math.max(1, Math.floor(n/48)), pts = [];
  for (let i=0;i<n;i+=step) pts.push(vals[i]);
  if (pts[pts.length-1] !== vals[n-1]) pts.push(vals[n-1]);
  const mn = Math.min(...pts), mx = Math.max(...pts), rng = (mx-mn)||1;
  const d = pts.map((v,i) => (i?'L':'M') + (i/(pts.length-1)*52).toFixed(1) + ' ' + (15-(v-mn)/rng*13).toFixed(1)).join(' ');
  return '<svg width="52" height="17" viewBox="0 0 52 17" aria-hidden="true"><path d="' + d +
         '" fill="none" stroke="' + stroke + '" stroke-width="1.3"/></svg>';
}
function renderRail(){
  const rail = $('#rail'); rail.innerHTML = '';
  const list = shown().map(c => [c, win(c,'w')]).sort((a,b) => b[1].total - a[1].total);
  const count = (list.length > RAIL_MAX ? 'top ' + RAIL_MAX + ' of ' + list.length : list.length + ' companies');
  rail.appendChild(el('div','railhead', count + ' · ' + state.period));
  $('#dockLabel').textContent = count + ' · sectors, map layers';
  list.slice(0, RAIL_MAX).forEach(([c,w]) => {
    const b = el('button','card');
    b.style.setProperty('--c', color(c));
    b.setAttribute('aria-current', c.ticker === state.active);
    b.innerHTML = '<span class="tkr">' + c.ticker + '</span>' +
      '<span class="chg ' + cls(w.total) + '">' + spct(w.total) + '</span>' +
      '<span class="nm">' + c.name + '</span>' + sparkline(w.close, w.total>=0 ? '#3DDC97' : '#FF5F6D');
    b.onclick = () => select(c.ticker, true);
    rail.appendChild(b);
  });
}

/* ============================ charts ============================ */
const AXIS = { axisLine:{lineStyle:{color:'rgba(122,152,204,.35)'}},
  axisLabel:{color:'#8CA1C4',fontSize:10,fontFamily:'IBM Plex Mono'},
  splitLine:{lineStyle:{color:'rgba(122,152,204,.10)'}} };
const ax = o => Object.assign({}, AXIS, o);
const money = ax({ type:'value', scale:true, axisLabel:{color:'#8CA1C4',fontSize:10,fontFamily:'IBM Plex Mono',formatter:v=>'$'+v.toFixed(0)} });
const percent = ax({ type:'value', axisLabel:{color:'#8CA1C4',fontSize:10,fontFamily:'IBM Plex Mono',formatter:v=>v.toFixed(0)+'%'} });
const BASE = { backgroundColor:'transparent', textStyle:{color:'#E7EEFB',fontFamily:'Archivo'}, animationDuration:400,
  tooltip:{ backgroundColor:'rgba(11,19,37,.96)', borderColor:'rgba(122,152,204,.35)',
            textStyle:{color:'#E7EEFB',fontSize:11,fontFamily:'IBM Plex Mono'} },
  grid:{ left:48, right:16, top:18, bottom:26 } };
const tip = o => Object.assign({}, BASE.tooltip, o);
const charts = new Map();
function chart(id, option){
  const node = document.getElementById(id); if (!node) return;
  let c = charts.get(id);
  if (c && !c.isDisposed() && c.getDom() !== node){ c.dispose(); c = null; }
  if (!c || c.isDisposed()){ c = echarts.init(node, null, { renderer:'canvas' }); charts.set(id, c); }
  c.setOption(Object.assign({}, BASE, option), true);
  return c;
}
addEventListener('resize', () => charts.forEach(c => !c.isDisposed() && c.resize()));

function priceChart(c, w){
  const bb = bollinger(w.close), s50 = sma(w.close,50), s200 = sma(w.close,200);
  const series = [
    { name:'Price', type:'line', data:w.close, showSymbol:false, lineStyle:{width:2,color:color(c)},
      areaStyle:{ color:{ type:'linear',x:0,y:0,x2:0,y2:1,
        colorStops:[{offset:0,color:color(c)+'44'},{offset:1,color:color(c)+'00'}] } } },
    { name:'Volume (M)', type:'bar', yAxisIndex:1, data:w.volume, itemStyle:{color:'rgba(122,152,204,.22)'} }
  ];
  if (state.overlays.has('bb')){
    series.push({ name:'Upper band', type:'line', data:bb.up, showSymbol:false, lineStyle:{width:1,type:'dashed',color:'rgba(159,179,209,.55)'} });
    series.push({ name:'Lower band', type:'line', data:bb.lo, showSymbol:false, lineStyle:{width:1,type:'dashed',color:'rgba(159,179,209,.55)'} });
  }
  if (state.overlays.has('sma')){
    series.push({ name:'SMA 50', type:'line', data:s50, showSymbol:false, lineStyle:{width:1.4,color:'#F2C14E'} });
    series.push({ name:'SMA 200', type:'line', data:s200, showSymbol:false, lineStyle:{width:1.4,color:'#FF8FA3'} });
  }
  chart('c-price', { grid:{left:52,right:42,top:16,bottom:28}, tooltip:tip({trigger:'axis'}),
    xAxis:ax({type:'category',data:w.dates,axisTick:{show:false}}),
    yAxis:[ money, { type:'value', show:true, max: Math.max(...w.volume.filter(v => v != null))*4, axisLabel:{show:false},
                     splitLine:{show:false}, axisLine:{show:false} } ],
    series });
}
function returnsChart(c, w){
  const a = cumulative(w.rets).map(v => (v-1)*100), b = cumulative(w.benchRets).map(v => (v-1)*100);
  chart('c-ret', { tooltip:tip({trigger:'axis',valueFormatter:v=>v.toFixed(1)+'%'}),
    legend:{ data:[c.ticker, DATA.benchmark], textStyle:{color:'#8CA1C4',fontSize:10}, top:0, right:0 },
    xAxis:ax({type:'category',data:w.dates,axisTick:{show:false}}), yAxis:percent,
    series:[ { name:c.ticker, type:'line', data:a, showSymbol:false, lineStyle:{width:2.2,color:color(c)} },
             { name:DATA.benchmark, type:'line', data:b, showSymbol:false, lineStyle:{width:1.6,color:'#9FB3D1',type:'dashed'} } ] });
}
function momentumCharts(c, w){
  const m = macd(w.close), r = rsi(w.close);
  chart('c-macd', { grid:{left:48,right:16,top:16,bottom:24}, tooltip:tip({trigger:'axis'}),
    xAxis:ax({type:'category',data:w.dates,axisTick:{show:false}}), yAxis:ax({type:'value'}),
    series:[
      { name:'Histogram', type:'bar', data:m.hist.map(v => ({ value:v,
          itemStyle:{ color: v>=0 ? 'rgba(61,220,151,.55)' : 'rgba(255,95,109,.55)' } })) },
      { name:'MACD', type:'line', data:m.line, showSymbol:false, lineStyle:{width:1.5,color:'#4FD3E8'} },
      { name:'Signal', type:'line', data:m.sig, showSymbol:false, lineStyle:{width:1.5,color:'#F2C14E'} } ] });
  chart('c-rsi', { grid:{left:48,right:16,top:14,bottom:24},
    tooltip:tip({trigger:'axis',valueFormatter:v=>v==null?'—':(+v).toFixed(1)}),
    xAxis:ax({type:'category',data:w.dates,axisTick:{show:false}}),
    yAxis:ax({type:'value',min:0,max:100,interval:25}),
    series:[{ name:'RSI', type:'line', data:r, showSymbol:false, lineStyle:{width:1.6,color:'#A78BFA'},
      markLine:{ silent:true, symbol:'none', label:{show:false},
        lineStyle:{color:'rgba(159,179,209,.35)',type:'dotted'}, data:[{yAxis:70},{yAxis:30}] } }] });
}
function riskCharts(c, w){
  const cum = cumulative(w.rets); let peak = 0;
  const dd = cum.map(v => { peak = Math.max(peak, v); return (v/peak-1)*100; });
  chart('c-dd', { tooltip:tip({trigger:'axis',valueFormatter:v=>v.toFixed(1)+'%'}),
    xAxis:ax({type:'category',data:w.dates,axisTick:{show:false}}), yAxis:percent,
    series:[{ type:'line', data:dd, showSymbol:false, lineStyle:{width:1.4,color:'#FF5F6D'},
      areaStyle:{color:'rgba(255,95,109,.22)'} }] });

  const months = {};
  w.dates.forEach((d,i) => { const k = d.slice(0,7); (months[k] = months[k] || []).push(w.close[i]); });
  const keys = Object.keys(months).sort();
  const years = [...new Set(keys.map(k => k.slice(0,4)))];
  const cells = keys.map(k => { const a = months[k];
    return [ +k.slice(5,7)-1, years.indexOf(k.slice(0,4)), +((a[a.length-1]/a[0]-1)*100).toFixed(2) ]; });
  const lim = Math.max(6, ...cells.map(x => Math.abs(x[2])));
  const MON = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'];
  chart('c-heat', { grid:{left:48,right:16,top:10,bottom:40},
    tooltip:tip({ formatter:p => MON[p.data[0]] + ' ' + years[p.data[1]] + ': ' + (p.data[2]>=0?'+':'') + p.data[2] + '%' }),
    xAxis:ax({type:'category',data:MON,splitArea:{show:false}}),
    yAxis:ax({type:'category',data:years,splitArea:{show:false}}),
    visualMap:{ min:-lim, max:lim, show:true, orient:'horizontal', left:'center', bottom:0, calculable:false,
      itemWidth:10, itemHeight:70, textStyle:{color:'#8CA1C4',fontSize:10},
      inRange:{ color:['#FF5F6D','rgba(18,26,46,.9)','#3DDC97'] } },
    series:[{ type:'heatmap', data:cells, itemStyle:{borderColor:'rgba(3,5,11,.8)',borderWidth:1} }] });
}

/* ============================ drawer ============================ */
const DTABS = [['price','Price'],['returns','Returns'],['momentum','Momentum'],['risk','Risk'],['profile','Company']];
function select(ticker, fly){
  const c = BY_T[ticker]; if (!c) return;
  state.active = ticker;
  if (MOBILE()) dock.classList.remove('open'), $('#dockToggle').setAttribute('aria-expanded','false');
  if (fly) flyToHQ(c);
  openDrawer(c);
  renderRail();
  if (state.view === 'board') renderBoard();
}
function closeDrawer(){ state.active = null; $('#drawer').classList.remove('open'); renderRail(); }
function openDrawer(c){
  const w = win(c), full = seriesOf(c, null);
  const valid = full.close.filter(v => v != null);
  const last = valid[valid.length-1], prev = valid[valid.length-2];
  const move = last/prev - 1;
  const hd = $('#dhead');
  hd.style.setProperty('--c', color(c));
  hd.innerHTML = '<button class="x" aria-label="Close">&times;</button>' +
    '<div class="dsector">' + c.sector + ' · ' + c.ticker + '</div>' +
    '<h2 class="dname">' + c.name + '</h2>' +
    '<div class="dwhere">' + (c.hq ? c.hq.city : '—') +
      (c.profile.employees ? ' · ' + c.profile.employees.toLocaleString() + ' employees' : '') + '</div>' +
    '<div class="dprice"><span class="p num">' + usd(last) + '</span>' +
      '<span class="num ' + cls(move) + '">' + spct(move,2) + (w.tag === 'daily' ? ' today' : ' this week') + '</span>' +
      '<span class="num ' + cls(w.total) + '" style="margin-left:auto">' + spct(w.total) + ' · ' + state.period + '</span></div>';
  hd.querySelector('.x').onclick = closeDrawer;
  const dt = $('#dtabs'); dt.innerHTML = ''; dt.style.setProperty('--c', color(c));
  DTABS.forEach(([k,label]) => {
    const b = el('button', null, label);
    b.setAttribute('aria-selected', k === state.dtab);
    b.onclick = () => { state.dtab = k; openDrawer(c); };
    dt.appendChild(b);
  });
  $('#drawer').classList.add('open');
  renderDrawerBody(c, w);
}
function renderDrawerBody(c, w){
  const b = $('#dbody'); b.innerHTML = '';
  const resNote = w.tag === 'weekly'
    ? '<div class="note">weekly bars — daily history is loaded for the detail tier only</div>' : '';
  if (state.dtab === 'price'){
    b.appendChild(el('div','toggles',
      '<button data-o="sma" aria-pressed="' + state.overlays.has('sma') + '">Moving averages</button>' +
      '<button data-o="bb" aria-pressed="' + state.overlays.has('bb') + '">Bollinger bands</button>'));
    b.querySelectorAll('[data-o]').forEach(btn => btn.onclick = () => {
      const k = btn.dataset.o;
      state.overlays.has(k) ? state.overlays.delete(k) : state.overlays.add(k);
      renderDrawerBody(c, w);
    });
    b.insertAdjacentHTML('beforeend', resNote);
    b.appendChild(el('div','chartwrap','<h4>Price and volume · ' + state.period + '</h4><div class="chart tall" id="c-price"></div>'));
    b.appendChild(rangeBlock(w));
    priceChart(c, w);
  }
  if (state.dtab === 'returns'){
    const bw = win(BENCH, w.tag === 'daily' ? 'd' : 'w');
    b.appendChild(kpiBlock([
      ['Return · ' + state.period, spct(w.total), cls(w.total)],
      ['Annualized', spct(w.annRet), cls(w.annRet)],
      ['vs ' + DATA.benchmark, spct(w.total - bw.total), cls(w.total - bw.total)] ]));
    b.appendChild(el('div','chartwrap','<h4>Cumulative return against the benchmark</h4><div class="chart tall" id="c-ret"></div>'));
    returnsChart(c, w);
  }
  if (state.dtab === 'momentum'){
    const m = macd(w.close), r = rsi(w.close);
    const lastR = [...r].reverse().find(v => v != null), h = m.hist[m.hist.length-1];
    const s50 = sma(w.close,50)[w.close.length-1];
    b.appendChild(kpiBlock([
      ['RSI 14', num(lastR,1), lastR > 70 ? 'down' : lastR < 30 ? 'up' : ''],
      ['MACD histogram', num(h,2), cls(h)],
      ['Trend', s50 ? (w.last > s50 ? 'Above 50' : 'Below 50') : '—', ''] ]));
    b.insertAdjacentHTML('beforeend', resNote);
    b.appendChild(el('div','chartwrap','<h4>MACD 12 / 26 / 9</h4><div class="chart" id="c-macd"></div>'));
    b.appendChild(el('div','chartwrap','<h4>Relative strength · 70 and 30 marked</h4><div class="chart" id="c-rsi"></div>'));
    momentumCharts(c, w);
  }
  if (state.dtab === 'risk'){
    b.appendChild(kpiBlock([
      ['Volatility', pct(w.annVol), ''], ['Sharpe', num(w.sharpe), cls(w.sharpe)], ['Sortino', num(w.sortino), cls(w.sortino)],
      ['Max drawdown', pct(w.maxDD), 'down'], ['Beta', num(w.beta), ''], ['Alpha', spct(w.alpha), cls(w.alpha)] ]));
    b.appendChild(el('div','chartwrap','<h4>Drawdown from running peak</h4><div class="chart" id="c-dd"></div>'));
    b.appendChild(el('div','chartwrap','<h4>Monthly returns</h4><div class="chart tall" id="c-heat"></div>'));
    riskCharts(c, w);
  }
  if (state.dtab === 'profile'){
    const p = c.profile;
    b.appendChild(kpiBlock([
      ['Market cap', cap(p.marketCap), ''], ['P/E trailing', num(p.pe,1), ''], ['Forward P/E', num(p.forwardPe,1), ''],
      ['Dividend yield', p.divYield ? pct(p.divYield) : '—', ''], ['Avg volume', vlm(p.avgVolume), ''],
      ['Beta · 5y', num(p.beta), ''] ]));
    const dl = el('dl','deflist');
    dl.innerHTML = '<dt>Headquarters</dt><dd>' + (c.hq && c.hq.address ? c.hq.address : (c.hq ? c.hq.city : '—')) + '</dd>' +
      '<dt>City</dt><dd>' + (c.hq ? c.hq.city : '—') + '</dd>' +
      '<dt>Coordinates</dt><dd>' + (c.hq ? c.hq.lat.toFixed(3) + ', ' + c.hq.lon.toFixed(3) : '—') + '</dd>' +
      '<dt>Industry</dt><dd style="font-family:var(--sans)">' + (c.industry || c.sector) + '</dd>' +
      (p.website ? '<dt>Site</dt><dd><a style="color:var(--beam)" href="' + p.website + '" target="_blank" rel="noopener">' +
        p.website.replace(/^https?:\/\//,'') + '</a></dd>' : '');
    b.appendChild(dl);
    const fly = el('button','ghost','Fly to headquarters');
    fly.style.marginBottom = '12px';
    fly.onclick = () => { setView('globe'); flyToHQ(c); };
    b.appendChild(fly);
    if (p.summary) b.appendChild(el('p','prose', p.summary));
  }
}
function kpiBlock(items){
  const g = el('div','kpis');
  items.forEach(([k,v,c]) => g.appendChild(el('div','kpi','<div class="k">' + k + '</div><div class="v ' + (c||'') + '">' + v + '</div>')));
  return g;
}
function rangeBlock(w){
  const d = el('div','range');
  const pos = Math.max(0, Math.min(1, (w.last - w.lo) / ((w.hi - w.lo) || 1)));
  d.innerHTML = '<div class="rangebar"><i style="left:' + (pos*100).toFixed(1) + '%"></i></div>' +
    '<div class="rangelabels"><span>' + usd(w.lo) + ' low</span><span>' + state.period +
    ' range</span><span>' + usd(w.hi) + ' high</span></div>';
  return d;
}

/* ============================ board ============================ */
const COLS = [
  ['name','Company',      (c) => '<span class="badge" style="--c:' + color(c) + '"></span>' + c.name],
  ['ticker','Ticker',     (c) => c.ticker],
  ['sector','Sector',     (c) => '<span style="font-family:var(--sans);color:var(--muted)">' + c.sector + '</span>'],
  ['last','Price',        (c,w) => usd(w.last)],
  ['total','Return',      (c,w) => '<span class="' + cls(w.total) + '">' + spct(w.total) + '</span>'],
  ['annRet','Annualized', (c,w) => '<span class="' + cls(w.annRet) + '">' + spct(w.annRet) + '</span>'],
  ['annVol','Volatility', (c,w) => pct(w.annVol)],
  ['sharpe','Sharpe',     (c,w) => '<span class="' + cls(w.sharpe) + '">' + num(w.sharpe) + '</span>'],
  ['maxDD','Max DD',      (c,w) => '<span class="down">' + pct(w.maxDD) + '</span>'],
  ['beta','Beta',         (c,w) => num(w.beta)],
  ['corrBench','Corr ' + DATA.benchmark, (c,w) => num(w.corrBench)],
  ['marketCap','Market cap', (c) => cap(c.profile.marketCap)]
];
const sortValue = (c, w, key) =>
  key === 'name' ? c.name : key === 'ticker' ? c.ticker : key === 'sector' ? c.sector :
  key === 'marketCap' ? (c.profile.marketCap || 0) : w[key];

function renderBoard(){
  const rows = shown().map(c => [c, win(c,'w')]);
  rows.sort((a,b) => {
    const va = sortValue(a[0], a[1], state.sortKey), vb = sortValue(b[0], b[1], state.sortKey);
    const na = (va == null || (typeof va === 'number' && !isFinite(va)));
    const nb = (vb == null || (typeof vb === 'number' && !isFinite(vb)));
    if (na || nb) return na && nb ? 0 : na ? 1 : -1;
    return (va > vb ? 1 : va < vb ? -1 : 0) * state.sortDir;
  });
  const visible = rows.slice(0, state.rows);
  $('#leadhint').textContent = 'Showing ' + visible.length + ' of ' + rows.length +
    ' · click a column to re-sort, a row to open the company · statistics use weekly bars so every company is comparable';

  const thead = $('#lead thead'); thead.innerHTML = '';
  const tr = el('tr');
  COLS.forEach(([key,label]) => {
    const th = el('th', null, label + (state.sortKey === key ? (state.sortDir < 0 ? ' ▾' : ' ▴') : ''));
    if (state.sortKey === key) th.setAttribute('data-active','');
    th.onclick = () => { if (state.sortKey === key) state.sortDir *= -1; else { state.sortKey = key; state.sortDir = -1; } renderBoard(); };
    tr.appendChild(th);
  });
  thead.appendChild(tr);
  const tb = $('#lead tbody'); tb.innerHTML = '';
  visible.forEach(([c,w]) => {
    const r = el('tr');
    COLS.forEach(([key,,fmt]) => r.appendChild(el('td', key === 'name' ? 'name' : null, fmt(c,w))));
    if (c.ticker === state.active) r.style.background = 'rgba(122,152,204,.12)';
    r.onclick = () => select(c.ticker, false);
    tb.appendChild(r);
  });

  /* scatter */
  const labelled = rows.length <= 60;
  chart('c-scatter', { grid:{left:56,right:26,top:20,bottom:44},
    tooltip:tip({ formatter:p => p.data.n + '<br/>vol ' + p.data.value[0].toFixed(1) + '% · ret ' +
                  p.data.value[1].toFixed(1) + '%<br/>Sharpe ' + p.data.s.toFixed(2) }),
    xAxis:ax({type:'value',name:'volatility %',nameLocation:'middle',nameGap:26,nameTextStyle:{color:'#5C6E90',fontSize:10},scale:true}),
    yAxis:ax({type:'value',name:'annual return %',nameLocation:'middle',nameGap:40,nameTextStyle:{color:'#5C6E90',fontSize:10},scale:true}),
    series:[
      { type:'scatter', large:true, data: rows.map(([c,w]) => ({
          value:[w.annVol*100, w.annRet*100], n:c.name, s:w.sharpe,
          symbolSize: Math.max(7, Math.min(34, Math.sqrt((c.profile.marketCap||2e10)/1e9)*2.4)),
          itemStyle:{ color:color(c)+'99', borderColor:color(c), borderWidth:1 },
          label:{ show:labelled, formatter:c.ticker, position:'right', color:'#8CA1C4', fontSize:10, fontFamily:'IBM Plex Mono' } })) },
      { type:'scatter', symbol:'diamond', symbolSize:16,
        data:[{ value:[win(BENCH,'w').annVol*100, win(BENCH,'w').annRet*100], n:BENCH.name, s:win(BENCH,'w').sharpe,
          itemStyle:{color:'#9FB3D1'},
          label:{show:true,formatter:DATA.benchmark,position:'right',color:'#9FB3D1',fontSize:10,fontFamily:'IBM Plex Mono'} }] } ] });

  /* correlation of the top slice */
  const CORR_N = 16;
  const top = visible.slice(0, CORR_N).map(r => r[0]).concat([BENCH]);
  const ts = top.map(c => c.ticker);
  const cells = [];
  for (let i=0;i<top.length;i++) for (let j=0;j<top.length;j++)
    cells.push([j, i, +(i===j ? 1 : corr(win(top[i],'w').rets, win(top[j],'w').rets)).toFixed(2)]);
  $('#corrhint').textContent = 'Weekly-return correlation across the top ' + Math.min(CORR_N, visible.length) +
    ' rows of the current sort, plus ' + DATA.benchmark + '.';
  chart('c-corr', { grid:{left:60,right:14,top:14,bottom:54},
    tooltip:tip({ formatter:p => ts[p.data[1]] + ' / ' + ts[p.data[0]] + ': ' + p.data[2] }),
    xAxis:ax({type:'category',data:ts,axisLabel:{color:'#8CA1C4',fontSize:9,fontFamily:'IBM Plex Mono',rotate:52}}),
    yAxis:ax({type:'category',data:ts,axisLabel:{color:'#8CA1C4',fontSize:9,fontFamily:'IBM Plex Mono'},inverse:true}),
    visualMap:{ min:0, max:1, orient:'horizontal', left:'center', bottom:2, itemWidth:10, itemHeight:80,
      textStyle:{color:'#8CA1C4',fontSize:10}, inRange:{ color:['rgba(18,26,46,.9)','#4FD3E8','#F2C14E'] } },
    series:[{ type:'heatmap', data:cells, itemStyle:{borderColor:'rgba(3,5,11,.75)',borderWidth:1},
      label:{ show: top.length <= 12, color:'#0A1021', fontSize:9, fontFamily:'IBM Plex Mono' } }] });

  /* equal-weight basket */
  const bw = win(BENCH,'w');
  const len = Math.min(...rows.map(r => r[1].rets.length));
  const pr = [];
  for (let i=0;i<len;i++){ let s = 0;
    for (const [,w] of rows) s += w.rets[w.rets.length - len + i];
    pr.push(s / rows.length); }
  const pc = cumulative(pr).map(v => (v-1)*100), bc = cumulative(bw.rets).map(v => (v-1)*100);
  const pRet = mean(pr)*52, pVol = sd(pr)*Math.sqrt(52);
  $('#porthint').textContent = rows.length + ' holdings · ' + spct(pRet) + ' annualized · ' + pct(pVol) +
    ' volatility · Sharpe ' + num((pRet - DATA.riskFree)/pVol) + ' · benchmark Sharpe ' + num(bw.sharpe);
  chart('c-port', { tooltip:tip({trigger:'axis',valueFormatter:v=>v.toFixed(1)+'%'}),
    legend:{ data:['Selection','Benchmark'], textStyle:{color:'#8CA1C4',fontSize:10}, top:0, right:0 },
    xAxis:ax({type:'category',data:bw.dates.slice(-pc.length),axisTick:{show:false}}), yAxis:percent,
    series:[
      { name:'Selection', type:'line', data:pc, showSymbol:false, lineStyle:{width:2.4,color:'#4FD3E8'},
        areaStyle:{ color:{ type:'linear',x:0,y:0,x2:0,y2:1,
          colorStops:[{offset:0,color:'rgba(79,211,232,.28)'},{offset:1,color:'rgba(79,211,232,0)'}] } } },
      { name:'Benchmark', type:'line', data:bc, showSymbol:false, lineStyle:{width:1.6,color:'#9FB3D1',type:'dashed'} } ] });

  /* sector weights */
  const bySector = {};
  rows.forEach(([c]) => { bySector[c.sector] = (bySector[c.sector] || 0) + (c.profile.marketCap || 0); });
  chart('c-cap', { tooltip:tip({ formatter:p => p.name + '<br/>' + cap(p.value) + ' · ' + p.percent.toFixed(1) + '%' }),
    series:[{ type:'pie', radius:['52%','78%'], center:['50%','52%'],
      itemStyle:{ borderColor:'rgba(3,5,11,.9)', borderWidth:2 },
      label:{ color:'#8CA1C4', fontSize:10, formatter:'{b}' },
      data: Object.entries(bySector).sort((a,b) => b[1]-a[1])
        .map(([s,v]) => ({ name:s, value:v, itemStyle:{ color: PALETTE[s] || '#9FB3D1' } })) }] });
}

/* ============================ boot ============================ */
function renderAll(){
  _cache.clear();
  renderRail();
  if (map.getSource && map.getSource('hq')) refreshPoints();
  if (state.view === 'board') renderBoard();
  if (state.active && BY_T[state.active]) openDrawer(BY_T[state.active]);
}
addEventListener('keydown', e => {
  if (e.target.tagName === 'INPUT') { if (e.key === 'Escape') e.target.blur(); return; }
  if (e.key === 'Escape') closeDrawer();
  if (e.key === 'g') setView('globe');
  if (e.key === 'b') setView('board');
  if (e.key === 'f') $('#fs').click();
  if (e.key === '/') { e.preventDefault(); $('#q').focus(); }
});
renderAll();
setTimeout(() => toast('Click a dot or a cluster · f for full screen · b for the board'), 1400);
</script>
</body>
</html>
'''

print(f'template: {len(TEMPLATE)/1024:,.0f} KB')

In [ ]:
from IPython.display import HTML, display

html = TEMPLATE.replace('/*__PAYLOAD__*/', payload_json)
with open(OUT_HTML, 'w', encoding='utf-8') as f:
    f.write(html)
print(f'wrote {OUT_HTML} · {len(html)/1024/1024:.2f} MB')
print('Open that file directly for the best experience — the globe wants the whole screen.')

display(HTML(
    '<iframe srcdoc="' + html.replace('&', '&amp;').replace('"', '&quot;') + '" '
    'allow="fullscreen" allowfullscreen '
    'style="width:100%;height:880px;border:1px solid #16223c;border-radius:12px;background:#03050B">'
    '</iframe>'))

## Notes

**Keys.** Esri's World Imagery, Dark Gray Canvas, Topo and Reference tile services and the AWS
terrarium elevation tiles are all open — no token, no watermark, no sign-up. If a map ever asks for a
key, something has swapped the style out.

**Keyboard.** `f` full screen, `g` globe, `b` board, `/` search, `Esc` closes the drawer.

**Scaling.** `UNIVERSE = 'sp500'` with `DETAIL_N = 120` lands around 5 MB of HTML and stays smooth
because the markers are a clustered GeoJSON layer rather than 500 DOM nodes, and only the labels for
what's on screen get rendered. Push `DETAIL_N` higher for more daily history and a bigger file.

**Other universes.** Any ticker list works — swap `sp500_table()` for your own dataframe with
`ticker, name, sector, industry, hq` and everything downstream follows. The payload is a plain dict,
so the same template renders from a database query or a CSV just as easily.

*Prices from Yahoo Finance via yfinance. Index membership from Wikipedia. Basemaps © Esri and
contributors; elevation from AWS Terrain Tiles. Educational use — not investment advice.*